# Parkinson's disease versus essential tremor — standalone Colab notebook

This notebook contains the complete **MultiROCKET–HYDRA classification
pipeline** for the public **PADS v1.0.0** movement data. Upload this one
`.ipynb` file to [Google Colab](https://colab.research.google.com/), select
a **Python GPU runtime** for acceleration, and choose **Runtime → Run all**. No repository,
companion scripts, local data, Google Drive mount, or API key is needed.
Internet access is required for packages and public data.

The default experiment uses **276 Parkinson's disease (PD) participants
and 28 essential tremor (ET) participants**, all retained tasks and both
wrists, five outer folds, three inner folds, one repeat, and a final
full-cohort model. **HYDRA convolutions use an NVIDIA GPU when available**;
MultiROCKET, scaling, and the ridge classifier run on the CPU. The
notebook also works in a CPU runtime. Training time and any speedup
depend on the hardware; the first Numba compilation adds startup time.

**What “standalone” means:** all project code is embedded below. The
notebook creates its own working files and downloads only the necessary
official data (approximately **157 MB** of movement binaries plus small
metadata files). Package installation needs additional disk space.
Outputs are generated by your run; there are no prefilled performance
numbers in this notebook.

**Data attribution:** [PADS v1.0.0 on PhysioNet](https://physionet.org/content/parkinsons-disease-smartwatch/1.0.0/),
DOI [10.13026/m0w9-zx22](https://doi.org/10.13026/m0w9-zx22),
**CC BY-NC-SA 4.0**. Use the official citation instructions when publishing
work based on the data. This is a research classifier; its evaluation
does not establish clinical validity or a PD/ET state-of-the-art ranking.


## 1. Configuration

Keep the defaults to reproduce the project's specified model and
evaluation procedure. `BATCH_SIZE` controls transform memory and `N_JOBS`
controls CPU workers. Reducing these values retains the same model
specification. Changing kernel/group counts changes the model. Changing
fold counts changes the evaluation procedure.

`HYDRA_DEVICE = 'auto'` uses CUDA when available and otherwise uses the
CPU. Set it to `'cpu'` to retain the original CPU implementation, or
`'cuda'` to require a GPU and stop clearly if CUDA is unavailable.
The GPU path copies the fitted aeon kernels and channel selections;
it does not change the model definition or participant splits. Floating
point differences can change features and predictions near ties, so
GPU and CPU outputs are not guaranteed to be identical. TF32 convolution
approximation is disabled. A larger `BATCH_SIZE` can improve GPU use
if memory permits; reduce it if GPU memory is exhausted.

Each participant is one input of shape **(132, 976)**: channel × sample.
Channels encode task segment, wrist, sensor, and axis; they are not 132
simultaneous physical sensor axes. Exact diagnosis strings define the
binary labels **PD = 0, ET = 1**. Original label `2` includes other
diagnoses and is never used as an ET-only target.

By default the final cell requests a browser download of the result ZIP.
Set `AUTO_DOWNLOAD = False` to download it manually from Colab's Files
sidebar instead. Colab's
working disk is temporary: download the ZIP before the runtime ends.


In [ ]:
from pathlib import Path
import os
import sys
import csv
import hashlib
import html
import json
import platform
import shutil
import subprocess
import time
import zipfile
from datetime import datetime, timezone
from IPython.display import display, FileLink, HTML, Image, Markdown

WORK = (Path('/content') if Path('/content').is_dir() else Path.cwd()) / 'pd_et_colab'
DATA_ROOT = WORK / 'data' / 'pads' / '1.0.0'
ENV_DIR = WORK / '.venv'

SEED = 20261017
OUTER_FOLDS = 5
INNER_FOLDS = 3
REPEATS = 1
MR_KERNELS = 6250
HYDRA_GROUPS = 64
HYDRA_KERNELS = 8
HYDRA_DEVICE = 'auto'  # 'auto', 'cpu', or 'cuda'
ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
BATCH_SIZE = 16
N_JOBS = min(4, os.cpu_count() or 1)
DOWNLOAD_WORKERS = 4
FIT_FINAL_MODEL = True
RUN_SYNTHETIC_CHECKS = True
RUN_GPU_BENCHMARK = True
AUTO_DOWNLOAD = True

# Optional: after fitting the final model, set this to an already
# preprocessed .npy file of shape (N, 132, 976). None skips new inference.
NEW_INPUTS_NPY = None

WORK = WORK.expanduser().resolve()
DATA_ROOT = DATA_ROOT.expanduser().resolve()
ENV_DIR = ENV_DIR.expanduser().resolve()
WORK.mkdir(parents=True, exist_ok=True)
print('Working directory:', WORK)
print('Notebook Python:', platform.python_version())
print('CPU workers:', N_JOBS, '| transform batch size:', BATCH_SIZE)


## 2. Install an isolated environment and detect the GPU

The scientific packages run in a separate Python environment, so this
notebook does not need to replace NumPy in the active Colab kernel or
restart it. aeon is pinned to **1.6.0** because this pipeline uses that
version's HYDRA sparse scaler. Other packages are resolved within the
declared compatible ranges and their actual versions are saved in the
results. This is a portable environment specification, not the original
macOS environment lock. New platform/package resolutions can produce
small numerical differences.

On a Colab GPU runtime, setup installs CUDA-enabled PyTorch. It uses
Colab's working PyTorch build/index when available, with CUDA 12.6 as a
fallback. On a CPU runtime it uses the official CPU build. It also
replaces a previous CPU build when you rerun after switching to a GPU.
The actual device and GPU name are printed and recorded in the run.

The setup checks Python compatibility explicitly. If Colab changes its
default Python to an unsupported version, select a compatible runtime
before continuing. Dependency installation errors stop the notebook.


In [ ]:
def run_command(command, *, log_path=None):
    # Stream subprocess output and stop on any nonzero exit.
    environment = os.environ.copy()
    for variable in ('PYTHONPATH', 'PYTHONHOME'):
        environment.pop(variable, None)
    environment.update(
        PYTHONUNBUFFERED='1', MPLBACKEND='Agg',
        MPLCONFIGDIR=str(WORK / '.matplotlib'),
        OMP_NUM_THREADS=str(N_JOBS), MKL_NUM_THREADS=str(N_JOBS),
        OPENBLAS_NUM_THREADS=str(N_JOBS), NUMBA_NUM_THREADS=str(N_JOBS),
        NUMBA_CACHE_DIR=str(WORK / '.numba_cache'),
    )
    log = Path(log_path).open('w', encoding='utf-8') if log_path else None
    process = None
    try:
        process = subprocess.Popen(
            [str(value) for value in command], cwd=WORK, env=environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            if log:
                log.write(line)
                log.flush()
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        if process is not None and process.stdout is not None:
            process.stdout.close()
        if log:
            log.close()


if not (3, 11) <= sys.version_info[:2] < (3, 15):
    raise RuntimeError('This aeon 1.6.0 notebook requires Python 3.11–3.14.')
if not isinstance(N_JOBS, int) or not 1 <= N_JOBS <= (os.cpu_count() or 1):
    raise ValueError('N_JOBS must be a positive integer within the CPU count.')
if HYDRA_DEVICE not in ('auto', 'cpu', 'cuda'):
    raise ValueError("HYDRA_DEVICE must be 'auto', 'cpu', or 'cuda'.")

PYTHON = ENV_DIR / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
if not PYTHON.is_file():
    try:
        run_command([sys.executable, '-m', 'venv', ENV_DIR])
    except subprocess.CalledProcessError:
        # Some hosted Python installations omit ensurepip. virtualenv
        # supplies its own pip bootstrap without replacing kernel NumPy.
        run_command([sys.executable, '-m', 'pip', 'install', '--quiet',
                     'virtualenv>=20,<21'])
        run_command([sys.executable, '-m', 'virtualenv', ENV_DIR])
run_command([PYTHON, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'])
torch_probe = (
    'import json, torch; print(json.dumps({"version": torch.__version__, '
    '"cuda": torch.version.cuda, "available": torch.cuda.is_available()}))'
)
def inspect_torch(interpreter):
    probe = subprocess.run([str(interpreter), '-c', torch_probe],
                           text=True, capture_output=True, timeout=60)
    if probe.returncode:
        return {}
    try:
        return json.loads(probe.stdout.strip().splitlines()[-1])
    except (ValueError, IndexError):
        return {}

kernel_torch = inspect_torch(sys.executable)
gpu_hardware = bool(kernel_torch.get('available'))
if sys.platform.startswith('linux') and shutil.which('nvidia-smi'):
    gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15,
    )
    gpu_hardware = gpu_hardware or (gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip()))
if HYDRA_DEVICE == 'cuda' and not gpu_hardware:
    raise RuntimeError('Select a GPU runtime in Colab, or use HYDRA_DEVICE="auto" or "cpu".')
want_cuda = HYDRA_DEVICE != 'cpu' and gpu_hardware and sys.platform.startswith('linux')
torch_spec = 'torch>=2.2,<3'
torch_index = 'https://download.pytorch.org/whl/cpu'
if want_cuda:
    torch_index = 'https://download.pytorch.org/whl/cu126'
    # Prefer the exact working build supplied by the hosted runtime.
    if kernel_torch.get('available') and kernel_torch.get('cuda') and str(kernel_torch.get('version', '')).startswith('2.'):
        torch_spec = 'torch==' + kernel_torch['version']
        torch_index = 'https://download.pytorch.org/whl/cu' + kernel_torch['cuda'].replace('.', '')
installed_torch = inspect_torch(PYTHON)
torch_command = [PYTHON, '-m', 'pip', 'install', '--quiet', torch_spec]
if sys.platform.startswith('linux'):
    torch_command += ['--index-url', torch_index]
    installed_tag = ('cu' + installed_torch['cuda'].replace('.', '')) if installed_torch.get('cuda') else 'cpu'
    if installed_torch and installed_tag != torch_index.rsplit('/', 1)[-1]:
        torch_command += ['--upgrade', '--force-reinstall']
run_command(torch_command)
torch_install = {'requested': torch_spec, 'index_url': torch_index if sys.platform.startswith('linux') else None,
                 'gpu_hardware_detected': gpu_hardware, 'cuda_build_requested': want_cuda}
REQUIREMENTS = '''aeon==1.6.0
torch>=2.2,<3
joblib>=1.3,<2
threadpoolctl>=3.2,<4
matplotlib>=3.8,<4
'''
REQUIREMENTS = '\n'.join(line.strip() for line in REQUIREMENTS.splitlines() if line.strip()) + '\n'
(WORK / 'requirements-colab.txt').write_text(REQUIREMENTS, encoding='utf-8')
run_command([PYTHON, '-m', 'pip', 'install', '--quiet', '-r', WORK / 'requirements-colab.txt'])
run_command([PYTHON, '-m', 'pip', 'check'])
actual_torch = inspect_torch(PYTHON)
torch_install['installed'] = actual_torch
if sys.platform.startswith('linux'):
    actual_tag = ('cu' + actual_torch['cuda'].replace('.', '')) if actual_torch.get('cuda') else 'cpu'
    torch_install['index_url'] = 'https://download.pytorch.org/whl/' + actual_tag
(WORK / 'torch-install.json').write_text(json.dumps(torch_install, indent=2) + '\n')
if HYDRA_DEVICE == 'cuda' and not actual_torch.get('available'):
    raise RuntimeError('CUDA-enabled PyTorch cannot use the selected GPU; inspect the installation output.')
with (WORK / 'environment-lock.txt').open('w', encoding='utf-8') as stream:
    subprocess.run([str(PYTHON), '-m', 'pip', 'freeze'], check=True, stdout=stream)
run_command([PYTHON, '-c',
             "import aeon, torch, numpy; from importlib.metadata import version; "
             "print({p: version(p) for p in ['aeon','torch','numpy','scipy','scikit-learn','numba']}); "
             "print('CUDA available:', torch.cuda.is_available(), '| GPU:', "
             "torch.cuda.get_device_name(0) if torch.cuda.is_available() else None); "
             "assert aeon.__version__ == '1.6.0'"])
print('Isolated environment ready. No notebook restart is required.')


## 3. Materialize the embedded project code

The following cells contain the full Python sources. Running them writes
an importable `sota_pd_et` package inside the notebook's working
directory. Keeping the package name also makes the saved model loadable
later. These cells can remain collapsed during normal use.


In [ ]:
# Embedded file: sota_pd_et/__init__.py
embedded_source = r'''"""Participant-level MultiROCKET-HYDRA evaluation for PADS PD versus ET."""

__version__ = "1.0.0"
'''
destination = WORK / 'sota_pd_et/__init__.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/data.py
embedded_source = r'''"""Read the existing official PADS binaries without changing the source dataset."""

from __future__ import annotations

import csv
from dataclasses import dataclass
import hashlib
import json
from pathlib import Path
import re

import numpy as np


PROJECT_ROOT = Path(__file__).resolve().parents[1]
DEFAULT_DATA_ROOT = PROJECT_ROOT / "data" / "pads" / "1.0.0"
LABELS = {"Parkinson's": 0, "Essential Tremor": 1}
MOVEMENT_SHAPE = (132, 976)
RETAINED_SEGMENTS = (
    "Relaxed1", "Relaxed2", "RelaxedTask1", "RelaxedTask2", "StretchHold",
    "HoldWeight", "DrinkGlas", "CrossArms", "TouchNose", "Entrainment1",
    "Entrainment2",
)
CHANNEL_NAMES = tuple(
    f"{task}_{wrist}_{sensor}_{axis}"
    for task in RETAINED_SEGMENTS
    for wrist in ("LeftWrist", "RightWrist")
    for sensor in ("Accelerometer", "Gyroscope")
    for axis in ("X", "Y", "Z")
)


@dataclass
class Cohort:
    X: np.ndarray
    y: np.ndarray
    ids: np.ndarray
    rows: list[dict]
    provenance: dict


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_cohort(root: str | Path = DEFAULT_DATA_ROOT) -> Cohort:
    """Select ET by its condition string; original label 2 is NOT an ET target.

    X[i] contains one participant's entire (132, 976) movement representation.
    The 132 channels are task-segment/wrist/sensor/axis combinations, rather than
    132 simultaneously recorded physical axes. No sliding windows are created.
    """
    root = Path(root).expanduser().resolve()
    checksum_path = root / "SHA256SUMS.txt"
    checksums = {}
    for line in checksum_path.read_text().splitlines():
        if line.strip():
            digest, name = line.split(maxsplit=1)
            checksums[name.removeprefix("*").removeprefix("./")] = digest
    source_hashes = {}

    def checked(relative: str) -> Path:
        path = root / relative
        if not path.resolve().is_relative_to(root):
            raise ValueError(f"Path escapes the data directory: {relative}")
        if relative not in checksums:
            raise ValueError(f"File missing from the official checksum list: {relative}")
        actual = sha256_file(path)
        if actual != checksums[relative]:
            raise ValueError(f"SHA-256 mismatch: {relative}")
        source_hashes[relative] = actual
        return path

    with checked("preprocessed/file_list.csv").open(newline="") as stream:
        table = list(csv.DictReader(stream))
    if not table or not {"id", "condition"}.issubset(table[0]):
        raise ValueError("The official participant table needs id and condition columns.")
    if len({r["id"] for r in table}) != len(table):
        raise ValueError("Duplicate participant IDs in the official table.")
    selected = sorted((r for r in table if r["condition"] in LABELS), key=lambda r: r["id"])
    if not selected or {r["condition"] for r in selected} != set(LABELS):
        raise ValueError("Both published PD and ET conditions must be present.")
    # These two files define the shape and ordering of the supplied binary inputs.
    checked("scripts/run_preprocessing.py")
    checked("scripts/utils/data_handling.py")
    X = np.empty((len(selected), *MOVEMENT_SHAPE), dtype=np.float32)
    rows = []
    for index, participant in enumerate(selected):
        pid = participant["id"]
        if re.fullmatch(r"[0-9]{3}", pid) is None:
            raise ValueError(f"Unexpected participant ID: {pid!r}")
        patient = json.loads(checked(f"patients/patient_{pid}.json").read_text())
        if str(patient["id"]) != pid or patient["condition"] != participant["condition"]:
            raise ValueError(f"Participant table/JSON disagreement: {pid}")
        path = checked(f"preprocessed/movement/{pid}_ml.bin")
        if path.stat().st_size != int(np.prod(MOVEMENT_SHAPE)) * 4:
            raise ValueError(f"Expected a little-endian float32 {MOVEMENT_SHAPE} file: {path}")
        values = np.fromfile(path, dtype="<f4").reshape(MOVEMENT_SHAPE)
        if not np.isfinite(values).all():
            raise ValueError(f"Nonfinite movement samples: {pid}")
        X[index] = values
        condition = participant["condition"]
        rows.append({
            "participant_id": pid,
            "condition": condition,
            "target_et": LABELS[condition],
            # Audit flag only. This is never included in the model input.
            "pd_with_et_comment": condition == "Parkinson's" and
            "essential tremor" in participant.get("disease_comment", "").lower(),
        })
    return Cohort(
        X=X,
        y=np.array([r["target_et"] for r in rows], dtype=np.int64),
        ids=np.array([r["participant_id"] for r in rows]),
        rows=rows,
        provenance={
            "data_root": str(root),
            "dataset": "PADS v1.0.0",
            "dataset_doi": "https://doi.org/10.13026/m0w9-zx22",
            "checksum_list_sha256": sha256_file(checksum_path),
            "verified_source_sha256": source_hashes,
            "label_mapping": LABELS,
            "participant_count": len(rows),
            "condition_counts": {condition: sum(r["condition"] == condition for r in rows)
                                 for condition in LABELS},
            "movement_shape": list(X.shape),
            "channel_names": list(CHANNEL_NAMES),
            "input": "Author-supplied movement binaries only; no questionnaire or demographics.",
            "preprocessing": "Published gravity correction, segment/task selection and 48-sample crop.",
            "timestamp_handling": "Binaries use sample indices. Raw timestamps are not resampled or gap-filtered.",
            "pd_with_et_comment_ids": [r["participant_id"] for r in rows if r["pd_with_et_comment"]],
        },
    )
'''
destination = WORK / 'sota_pd_et/data.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/model.py
embedded_source = r'''"""MultiROCKET-HYDRA features and a class-balanced ridge classifier.

Uses the aeon 1.6.0 implementation, including its HYDRA sparse scaler.
Unlike aeon's ready-made classifier, regularization is chosen by explicitly
refitting the ENTIRE feature map inside each inner training fold. Real-valued
ridge margins are exposed for AUROC; no hard labels are called probabilities.
"""

from __future__ import annotations

from dataclasses import dataclass
from importlib.metadata import version

import numpy as np
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler


@dataclass(frozen=True)
class ModelConfig:
    mr_kernels: int = 6250
    hydra_groups: int = 64
    hydra_kernels: int = 8
    batch_size: int = 32
    n_jobs: int = 4
    hydra_device: str = "cpu"


class FeatureMap:
    """Fit HYDRA/MultiROCKET and their scalers on a training fold only."""

    def __init__(self, config: ModelConfig, seed: int):
        self.config = config
        self.seed = seed

    def set_hydra_device(self, requested: str):
        """Select CPU/CUDA without refitting kernels, scalers, or the classifier."""
        from .hydra_gpu import resolve_device

        resolved = resolve_device(requested)
        self._hydra_device_request_ = requested
        self.hydra_device_ = resolved
        self.__dict__.pop("_hydra_device_transform_", None)
        return self

    def __getstate__(self):
        # aeon's fitted kernels and the sparse scaler stay on CPU. The CUDA
        # adapter contains disposable copies and must never enter a model file.
        state = self.__dict__.copy()
        state.pop("_hydra_device_transform_", None)
        if state.get("hydra_device_") == "cuda":
            # A saved GPU-trained model can perform inference on a CPU runtime.
            state["_hydra_device_request_"] = "auto"
        return state

    @staticmethod
    def _validate(X):
        X = np.asarray(X, dtype=np.float32)
        if X.ndim != 3 or len(X) == 0 or X.shape[1] < 1 or X.shape[2] < 10:
            raise ValueError("Expected nonempty (participants, channels, timepoints>=10).")
        if not np.isfinite(X).all():
            raise ValueError("The model cannot accept NaN or infinite samples.")
        return np.ascontiguousarray(X)

    def _hydra_features(self, X):
        import torch
        from .hydra_gpu import HydraDeviceTransform, resolve_device

        requested = getattr(self, "_hydra_device_request_",
                            getattr(self.config, "hydra_device", "cpu"))
        self.hydra_device_ = resolve_device(requested)
        if self.hydra_device_ == "cuda":
            adapter = getattr(self, "_hydra_device_transform_", None)
            if adapter is None:
                adapter = HydraDeviceTransform(self.hydra_, "cuda")
                self._hydra_device_transform_ = adapter
            transform = adapter.transform
        else:
            # Preserve the original CPU implementation and numerical behavior.
            transform = self.hydra_.transform

        with torch.inference_mode():
            return torch.cat([
                transform(X[start:start + self.config.batch_size])
                for start in range(0, len(X), self.config.batch_size)
            ], dim=0)

    def _mr_features(self, X):
        return np.concatenate([
            self.multirocket_.transform(X[start:start + self.config.batch_size])
            for start in range(0, len(X), self.config.batch_size)
        ], axis=0)

    @staticmethod
    def _join(hydra, multirocket):
        # float64 stabilizes the dual ridge solve with many standardized features.
        Z = np.concatenate((hydra.detach().cpu().numpy(), multirocket), axis=1)
        Z = np.ascontiguousarray(Z, dtype=np.float64)
        if not np.isfinite(Z).all():
            raise ValueError("Nonfinite transformed features.")
        return Z

    def fit_transform(self, X):
        X = self._validate(X)
        if len(X) < 2:
            raise ValueError("At least two training participants are needed for HYDRA scaling.")
        if version("aeon") != "1.6.0":
            raise RuntimeError("Install aeon==1.6.0: this code pins its private HYDRA scaler API.")
        from aeon.classification.convolution_based._hydra import _SparseScaler
        from aeon.transformations.collection.convolution_based import HydraTransformer, MultiRocket

        self.input_shape_ = X.shape[1:]
        self.hydra_ = HydraTransformer(
            n_kernels=self.config.hydra_kernels,
            n_groups=self.config.hydra_groups,
            n_jobs=self.config.n_jobs,
            random_state=self.seed,
            output_type="tensor",
        )
        self.hydra_.fit(X)
        self.set_hydra_device(getattr(self.config, "hydra_device", "cpu"))
        counts = self._hydra_features(X)
        self.hydra_scaler_ = _SparseScaler()
        hydra = self.hydra_scaler_.fit_transform(counts)
        self.multirocket_ = MultiRocket(
            n_kernels=self.config.mr_kernels,
            normalise=False,
            n_jobs=self.config.n_jobs,
            random_state=self.seed,
        )
        self.multirocket_.fit(X)
        self.mr_scaler_ = StandardScaler()
        multirocket = self.mr_scaler_.fit_transform(self._mr_features(X))
        return self._join(hydra, multirocket)

    def transform(self, X):
        X = self._validate(X)
        if X.shape[1:] != self.input_shape_:
            raise ValueError(f"Expected input shape (*, {self.input_shape_}), got {X.shape}.")
        hydra = self.hydra_scaler_.transform(self._hydra_features(X))
        multirocket = self.mr_scaler_.transform(self._mr_features(X))
        return self._join(hydra, multirocket)


def fit_ridge(Z: np.ndarray, y: np.ndarray, alpha: float) -> RidgeClassifier:
    if not np.array_equal(np.unique(y), [0, 1]):
        raise ValueError("Each training fold must include PD=0 and ET=1.")
    classifier = RidgeClassifier(alpha=float(alpha), class_weight="balanced", solver="auto")
    classifier.fit(Z, y)
    if not np.array_equal(classifier.classes_, [0, 1]):
        raise RuntimeError("Unexpected class order; a positive margin must mean ET.")
    return classifier


@dataclass
class FittedModel:
    features: FeatureMap
    classifier: RidgeClassifier

    def decision_function(self, X):
        return np.asarray(self.classifier.decision_function(self.features.transform(X))).reshape(-1)

    def predict(self, X):
        # Matches sklearn's binary ridge rule, including a score exactly equal to zero.
        return (self.decision_function(X) > 0.0).astype(np.int64)
'''
destination = WORK / 'sota_pd_et/model.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/evaluate.py
embedded_source = r'''"""Run honest, participant-level nested cross-validation on the current PADS data.

Run from the project root: python -m sota_pd_et.evaluate --n-jobs 4 --fit-final
No training or evaluation occurs when this module is imported.
"""

from __future__ import annotations

import argparse
from dataclasses import asdict
from datetime import datetime, timezone
import gc
from importlib.metadata import version
import json
import os
from pathlib import Path
import platform
import sys
import time

import joblib
import numpy as np
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, precision_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from threadpoolctl import threadpool_limits

from .data import DEFAULT_DATA_ROOT, LABELS, PROJECT_ROOT, load_cohort, sha256_file
from .model import FeatureMap, FittedModel, ModelConfig, fit_ridge


PERFORMANCE_METRICS = (
    "balanced_accuracy", "accuracy", "roc_auc", "average_precision_et", "et_recall",
    "pd_recall", "et_precision", "f1_et", "macro_f1", "matthews_correlation",
)


def write_json(path: Path, value):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def write_csv(path: Path, rows: list[dict]):
    import csv

    if not rows:
        raise ValueError(f"Refusing to write an empty table: {path}")
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def derive_seed(seed: int, *parts: int) -> int:
    return int(np.random.SeedSequence([seed, *parts]).generate_state(1)[0])


def validate_split(train, test, y):
    train, test = np.asarray(train, dtype=int), np.asarray(test, dtype=int)
    if not len(train) or not len(test):
        raise ValueError("A training or validation/test split is empty.")
    if len(np.unique(train)) != len(train) or len(np.unique(test)) != len(test):
        raise ValueError("Duplicate participants inside a split.")
    if np.intersect1d(train, test).size:
        raise ValueError("Participant leakage between training and validation/test.")
    if train.min() < 0 or test.min() < 0 or max(train.max(), test.max()) >= len(y):
        raise ValueError("Split index outside the cohort.")
    if not np.array_equal(np.unique(y[train]), [0, 1]):
        raise ValueError("Both classes must occur in every training split.")
    if not np.array_equal(np.unique(y[test]), [0, 1]):
        raise ValueError("Both classes must occur in every validation/test split.")


def make_inner_splits(y, n_splits: int, seed: int):
    if n_splits < 2 or np.bincount(y, minlength=2).min() < n_splits:
        raise ValueError("Inner fold count exceeds the number of participants in a class.")
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(np.zeros(len(y)), y))
    seen = np.zeros(len(y), dtype=int)
    for train, valid in splits:
        validate_split(train, valid, y)
        seen[valid] += 1
    if not np.all(seen == 1):
        raise RuntimeError("Each inner participant must be validated exactly once.")
    return splits


def make_plan(y, ids, outer_folds: int, inner_folds: int, repeats: int, seed: int):
    if len(np.unique(ids)) != len(ids) or len(ids) != len(y):
        raise ValueError("Require exactly one uniquely identified row per participant.")
    if not np.array_equal(np.unique(y), [0, 1]):
        raise ValueError("The target must contain exactly PD=0 and ET=1.")
    if repeats < 1 or outer_folds < 2 or np.bincount(y).min() < outer_folds:
        raise ValueError("Invalid repeat count or outer fold count for this cohort.")
    plan = []
    for repeat in range(repeats):
        split_seed = derive_seed(seed, repeat)
        outer = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=split_seed)
        seen = np.zeros(len(y), dtype=int)
        for fold, (train, test) in enumerate(outer.split(np.zeros(len(y)), y)):
            validate_split(train, test, y)
            seen[test] += 1
            model_seed = derive_seed(seed, repeat, fold)
            inner = make_inner_splits(y[train], inner_folds, model_seed)
            plan.append({
                "repeat": repeat + 1,
                "fold": fold + 1,
                "model_seed": model_seed,
                "train_indices": train.tolist(),
                "test_indices": test.tolist(),
                "train_ids": ids[train].tolist(),
                "test_ids": ids[test].tolist(),
                "inner_splits": [{
                    "train_positions": a.tolist(),
                    "validation_positions": b.tolist(),
                    "train_ids": ids[train[a]].tolist(),
                    "validation_ids": ids[train[b]].tolist(),
                } for a, b in inner],
            })
        if not np.all(seen == 1):
            raise RuntimeError("Each participant must be tested exactly once per repeat.")
    return plan


def metrics(y, scores):
    y, scores = np.asarray(y), np.asarray(scores, dtype=float)
    if scores.shape != y.shape or not np.isfinite(scores).all():
        raise ValueError("Need one finite decision score per participant.")
    if not np.array_equal(np.unique(y), [0, 1]):
        raise ValueError("Metrics require both PD and ET.")
    predicted = (scores > 0.0).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(y, predicted, labels=[0, 1]).ravel())
    return {
        "balanced_accuracy": float(balanced_accuracy_score(y, predicted)),
        "accuracy": float(accuracy_score(y, predicted)),
        "roc_auc": float(roc_auc_score(y, scores)),
        "average_precision_et": float(average_precision_score(y, scores)),
        "et_recall": tp / (tp + fn),
        "pd_recall": tn / (tn + fp),
        "et_precision": float(precision_score(y, predicted, zero_division=0)),
        "f1_et": float(f1_score(y, predicted, zero_division=0)),
        "macro_f1": float(f1_score(y, predicted, average="macro", zero_division=0)),
        "matthews_correlation": float(matthews_corrcoef(y, predicted)),
        "n_pd": tn + fp,
        "n_et": tp + fn,
        "pd_correct": tn,
        "pd_called_et": fp,
        "et_called_pd": fn,
        "et_correct": tp,
        "confusion_matrix_true_rows_predicted_columns_pd_et": [[tn, fp], [fn, tp]],
    }


def choose_alpha(rows):
    """Maximize mean inner-fold balanced accuracy; ties favor larger alpha."""
    best = max(row["inner_mean_balanced_accuracy"] for row in rows)
    candidates = [r for r in rows if abs(r["inner_mean_balanced_accuracy"] - best) <= 1e-12]
    return max(candidates, key=lambda r: r["alpha"])["alpha"]


def tune_alpha(X, y, config, alphas, inner_splits, seed):
    """Refit features/scalers separately in each inner fold; reuse only across alphas."""
    oof_scores = np.full((len(alphas), len(y)), np.nan)
    fold_ba = np.empty((len(alphas), len(inner_splits)))
    for fold, (train, valid) in enumerate(inner_splits):
        validate_split(train, valid, y)
        print(f"    Inner {fold + 1}/{len(inner_splits)}: feature extraction", flush=True)
        feature_map = FeatureMap(config, derive_seed(seed, fold))
        Z_train = feature_map.fit_transform(X[train])
        Z_valid = feature_map.transform(X[valid])
        for index, alpha in enumerate(alphas):
            classifier = fit_ridge(Z_train, y[train], alpha)
            scores = classifier.decision_function(Z_valid)
            oof_scores[index, valid] = scores
            fold_ba[index, fold] = balanced_accuracy_score(y[valid], scores > 0.0)
        del feature_map, Z_train, Z_valid, classifier
        gc.collect()
    if not np.isfinite(oof_scores).all():
        raise RuntimeError("Missing/nonfinite inner out-of-fold scores.")
    rows = [{
        "alpha": float(alpha),
        "inner_mean_balanced_accuracy": float(np.mean(fold_ba[index])),
        "inner_std_balanced_accuracy": float(np.std(fold_ba[index], ddof=1)),
        "inner_pooled_balanced_accuracy": float(balanced_accuracy_score(y, oof_scores[index] > 0)),
        "inner_pooled_roc_auc": float(roc_auc_score(y, oof_scores[index])),
    } for index, alpha in enumerate(alphas)]
    selected = choose_alpha(rows)
    selected_index = list(alphas).index(selected)
    return selected, rows, oof_scores[selected_index]


def prediction_rows(ids, y, scores, **extra):
    return [{
        **extra,
        "participant_id": str(pid),
        "target_et": int(target),
        "decision_score_et": float(score),
        "predicted_et": int(score > 0.0),
    } for pid, target, score in zip(ids, y, scores, strict=True)]


def fit_full_model(X, y, config, alpha, seed):
    features = FeatureMap(config, seed)
    Z = features.fit_transform(X)
    return FittedModel(features, fit_ridge(Z, y, alpha))


def describe_values(values):
    values = np.asarray(values, dtype=float)
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=1)) if len(values) > 1 else None,
        "values": values.tolist(),
    }


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, default=DEFAULT_DATA_ROOT)
    parser.add_argument("--output", type=Path)
    parser.add_argument("--outer-folds", type=int, default=5)
    parser.add_argument("--inner-folds", type=int, default=3)
    parser.add_argument("--repeats", type=int, default=1)
    parser.add_argument("--seed", type=int, default=20261017)
    parser.add_argument("--n-jobs", type=int, default=min(4, os.cpu_count() or 1))
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--mr-kernels", type=int, default=6250)
    parser.add_argument("--hydra-groups", type=int, default=64)
    parser.add_argument("--hydra-kernels", type=int, default=8)
    parser.add_argument("--hydra-device", choices=("auto", "cpu", "cuda"), default="cpu",
                        help="HYDRA convolution device; MultiROCKET and ridge remain on CPU.")
    parser.add_argument("--alphas", type=float, nargs="+", default=np.logspace(-3, 3, 7).tolist())
    parser.add_argument("--fit-final", action="store_true", help="Tune/refit all participants after CV; save joblib model.")
    args = parser.parse_args(argv)
    if args.seed < 0 or args.seed > 2**32 - 1:
        parser.error("--seed must be an integer in [0, 2**32-1].")
    if args.n_jobs == 0 or args.n_jobs < -1 or args.batch_size < 1:
        parser.error("Use --n-jobs -1 or a positive integer, and a positive --batch-size.")
    if args.n_jobs > (os.cpu_count() or 1):
        parser.error("--n-jobs exceeds the available CPU count.")
    if args.mr_kernels < 84 or args.hydra_groups < 2 or args.hydra_groups % 2 or args.hydra_kernels < 1:
        parser.error("Need --mr-kernels >=84, a positive even --hydra-groups, and --hydra-kernels >=1.")
    if not args.alphas or any(not np.isfinite(a) or a <= 0 for a in args.alphas):
        parser.error("All ridge alpha candidates must be finite and strictly positive.")
    args.alphas = sorted(set(args.alphas))
    return args


def run(args):
    if version("aeon") != "1.6.0":
        raise RuntimeError("Install the versions in sota_pd_et/requirements.txt.")
    from .hydra_gpu import resolve_device
    import torch

    device = resolve_device(args.hydra_device)
    config = ModelConfig(args.mr_kernels, args.hydra_groups, args.hydra_kernels,
                         args.batch_size, args.n_jobs, args.hydra_device)
    print(f"HYDRA device: {device}; MultiROCKET/scalers/ridge: CPU", flush=True)
    print("Loading and SHA-256-verifying the current PD/ET cohort...", flush=True)
    cohort = load_cohort(args.data_root)
    X, y, ids = cohort.X, cohort.y, cohort.ids
    print(f"Participants: PD={sum(y == 0)}, ET={sum(y == 1)}; tensor={X.shape}", flush=True)
    plan = make_plan(y, ids, args.outer_folds, args.inner_folds, args.repeats, args.seed)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output = (args.output or PROJECT_ROOT / "artifacts" / "pd_et_mr_hydra" / stamp).expanduser().resolve()
    if output.is_relative_to(args.data_root.expanduser().resolve()):
        raise ValueError("Choose an output directory outside the source dataset.")
    # Do not mix predictions/configurations from different runs.
    output.mkdir(parents=True, exist_ok=False)
    versions = {p: version(p) for p in (
        "aeon", "numpy", "scipy", "pandas", "scikit-learn", "numba", "torch", "joblib", "threadpoolctl",
    )}
    manifest = {
        "status": "running",
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "command": sys.argv,
        "python": sys.version,
        "platform": platform.platform(),
        "package_versions": versions,
        "acceleration": {
            "hydra_device_requested": args.hydra_device,
            "hydra_device_used": device,
            "gpu_name": torch.cuda.get_device_name(0) if device == "cuda" else None,
            "torch_cuda_version": torch.version.cuda,
            "multirocket_device": "cpu",
            "scalers_and_ridge_device": "cpu",
            "numerics": "CUDA uses the same fitted CPU kernels/channel indices; float32 "
            "convolutions and scatter reductions may differ numerically from CPU. "
            "TF32 is disabled for the CUDA convolution path.",
        },
        "config": {**{k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},
                   "output": str(output), "model": asdict(config)},
        "source_code_sha256": {str(p.relative_to(PROJECT_ROOT)): sha256_file(p)
                               for p in sorted(Path(__file__).parent.rglob("*.py"))},
        "dataset": cohort.provenance,
        "protocol": {
            "unit": "One participant, with all task/wrist/sensor/axis channels retained together.",
            "primary_metric": "Balanced accuracy from outer out-of-fold class predictions.",
            "alpha_selection": "Maximum mean inner-fold balanced accuracy; ties choose larger alpha.",
            "decision_rule": "ET if ridge decision score > 0; PD otherwise. Threshold fixed in advance.",
            "class_weight": "n_train / (2 * class_count), recalculated within every ridge fit.",
            "feature_fit": "HYDRA/MultiROCKET and both scalers refitted in each inner/outer training fold.",
            "probabilities": "Not produced. Decision scores are signed, uncalibrated ridge margins.",
            "representation_limit": "Published preprocessing includes task selection from the original study.",
        },
    }
    write_json(output / "manifest.json", manifest)
    write_csv(output / "cohort.csv", cohort.rows)
    write_json(output / "splits.json", plan)
    all_predictions, fold_metrics, repeat_metrics = [], [], []
    oof = np.full((args.repeats, len(y)), np.nan)
    coverage = np.zeros((args.repeats, len(y)), dtype=int)
    started = time.perf_counter()
    try:
        for split in plan:
            repeat, fold = split["repeat"], split["fold"]
            train, test = np.array(split["train_indices"]), np.array(split["test_indices"])
            inner = [(np.array(s["train_positions"]), np.array(s["validation_positions"]))
                     for s in split["inner_splits"]]
            destination = output / f"repeat_{repeat:02d}_fold_{fold:02d}"
            destination.mkdir()
            print(f"Repeat {repeat}/{args.repeats}, outer fold {fold}/{args.outer_folds}", flush=True)
            fold_started = time.perf_counter()
            alpha, search, inner_scores = tune_alpha(X[train], y[train], config,
                                                     args.alphas, inner, split["model_seed"])
            write_csv(destination / "alpha_search.csv", search)
            write_csv(destination / "inner_oof_predictions.csv", prediction_rows(ids[train], y[train], inner_scores))
            print(f"    Selected alpha={alpha:g}; refitting outer training participants", flush=True)
            model = fit_full_model(X[train], y[train], config, alpha, split["model_seed"])
            scores = model.decision_function(X[test])
            result = {"repeat": repeat, "fold": fold, "selected_alpha": alpha,
                      "elapsed_seconds": time.perf_counter() - fold_started, **metrics(y[test], scores)}
            rows = prediction_rows(ids[test], y[test], scores, repeat=repeat, fold=fold)
            write_csv(destination / "test_predictions.csv", rows)
            write_json(destination / "metrics.json", result)
            fold_metrics.append(result)
            all_predictions.extend(rows)
            oof[repeat - 1, test] = scores
            coverage[repeat - 1, test] += 1
            print(f"    BA={result['balanced_accuracy']:.4f}, AUROC={result['roc_auc']:.4f}, "
                  f"ET recall={result['et_recall']:.4f}", flush=True)
            del model
            gc.collect()
        if not np.all(coverage == 1) or not np.isfinite(oof).all():
            raise RuntimeError("Each participant needs exactly one outer prediction per repeat.")
        for repeat in range(args.repeats):
            repeat_metrics.append({"repeat": repeat + 1, **metrics(y, oof[repeat])})
        summary = {
            "primary_metric": "balanced_accuracy",
            "positive_class": "Essential Tremor (ET=1); Parkinson's disease is PD=0.",
            "cohort_counts": cohort.provenance["condition_counts"],
            "repeat_metrics": repeat_metrics,
            "across_repeats": {key: describe_values([r[key] for r in repeat_metrics])
                               for key in PERFORMANCE_METRICS},
            "outer_fold_metrics": fold_metrics,
            "across_outer_folds_descriptive": {key: describe_values([r[key] for r in fold_metrics])
                                              for key in PERFORMANCE_METRICS},
            "always_predict_pd_baseline": metrics(y, -np.ones(len(y))),
            "average_precision_no_skill_reference": float(np.mean(y)),
            "interpretation": [
                "All reported model test predictions are from outer held-out participants.",
                "AUROC and AP use continuous margins. Scores are not probabilities.",
                "Repeat-level pooled AUROC/AP compare margins from separately fitted models; "
                "also inspect the per-fold ranking metrics.",
                "Fold/repeat standard deviations are descriptive, not confidence intervals; fits overlap.",
                "Repeating CV does not create more independent ET participants.",
                "This estimates the specified procedure on the published PADS representation; "
                "it establishes neither a PD/ET SOTA ranking nor external clinical validity.",
            ],
        }
        write_csv(output / "oof_predictions.csv", sorted(all_predictions,
                  key=lambda r: (r["repeat"], r["participant_id"])))
        write_csv(output / "fold_metrics.csv", [{k: v for k, v in r.items() if not isinstance(v, list)}
                                               for r in fold_metrics])
        write_json(output / "summary.json", summary)
        if args.fit_final:
            print("Tuning/refitting the optional final model on all participants...", flush=True)
            final_seed = derive_seed(args.seed, args.repeats, args.outer_folds)
            inner = make_inner_splits(y, args.inner_folds, final_seed)
            write_json(output / "final_inner_splits.json", [
                {"train_ids": ids[a].tolist(), "validation_ids": ids[b].tolist()} for a, b in inner
            ])
            alpha, search, final_inner_scores = tune_alpha(X, y, config, args.alphas, inner, final_seed)
            write_csv(output / "final_alpha_search.csv", search)
            write_csv(output / "final_inner_oof_predictions.csv", prediction_rows(ids, y, final_inner_scores))
            model = fit_full_model(X, y, config, alpha, final_seed)
            joblib.dump({
                "format_version": 1,
                "model": model,
                "input_shape": list(X.shape[1:]),
                "channel_names": cohort.provenance["channel_names"],
                "label_mapping": LABELS,
                "decision_threshold": 0.0,
                "selected_alpha": alpha,
                "training_ids": ids.tolist(),
                "package_versions": versions,
                "source_code_sha256": manifest["source_code_sha256"],
                "evaluation_summary": "summary.json",
                "note": "Final model trained on all participants. Its training performance is not evaluation.",
            }, output / "final_model.joblib", compress=3)
        manifest.update(status="complete", finished_at_utc=datetime.now(timezone.utc).isoformat(),
                        elapsed_seconds=time.perf_counter() - started)
        write_json(output / "manifest.json", manifest)
        print(f"\nSaved results: {output}", flush=True)
        for key in ("balanced_accuracy", "roc_auc", "et_recall", "pd_recall", "average_precision_et"):
            statistic = summary["across_repeats"][key]
            print(f"{key}: {statistic['mean']:.6f}", flush=True)
        return output
    except BaseException as error:
        manifest.update(status="failed", error=f"{type(error).__name__}: {error}",
                        elapsed_seconds=time.perf_counter() - started)
        write_json(output / "manifest.json", manifest)
        raise


def main(argv=None):
    args = parse_args(argv)
    # CV itself is sequential; parallelize each transform and bound BLAS oversubscription.
    workers = (os.cpu_count() or 1) if args.n_jobs == -1 else args.n_jobs
    with threadpool_limits(limits=workers):
        run(args)


if __name__ == "__main__":
    main()
'''
destination = WORK / 'sota_pd_et/evaluate.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/predict.py
embedded_source = r'''"""Predict from a saved final model and already-preprocessed participant arrays."""

from __future__ import annotations

import argparse
import csv
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", type=Path, required=True)
    parser.add_argument("--inputs", type=Path, required=True, help=".npy array of shape (N, 132, 976).")
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--hydra-device", choices=("auto", "cpu", "cuda"), default=None,
                        help="Override saved HYDRA runtime policy; no retraining is performed.")
    args = parser.parse_args(argv)
    if args.output.exists():
        parser.error("The output file already exists; choose a new path.")
    # Load only your own model artifacts: joblib uses Python pickle serialization.
    bundle = joblib.load(args.model)
    if bundle.get("format_version") != 1:
        raise ValueError("Unsupported model bundle format.")
    for package, expected in bundle["package_versions"].items():
        if version(package) != expected:
            raise RuntimeError(f"Restore the training environment: {package} must be {expected}.")
    X = np.load(args.inputs, allow_pickle=False)
    if X.ndim != 3 or list(X.shape[1:]) != bundle["input_shape"] or len(X) == 0:
        raise ValueError(f"Expected (N, {bundle['input_shape']}) with N>=1; got {X.shape}.")
    if not np.isfinite(X).all():
        raise ValueError("Input contains nonfinite samples.")
    if args.hydra_device is not None:
        bundle["model"].features.set_hydra_device(args.hydra_device)
    scores = bundle["model"].decision_function(X)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    with args.output.open("x", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=[
            "case_index", "decision_score_et", "predicted_et", "predicted_condition",
        ])
        writer.writeheader()
        for index, score in enumerate(scores):
            predicted = int(score > bundle["decision_threshold"])
            writer.writerow({"case_index": index, "decision_score_et": float(score),
                             "predicted_et": predicted,
                             "predicted_condition": "Essential Tremor" if predicted else "Parkinson's"})
    print(args.output.resolve())


if __name__ == "__main__":
    main()
'''
destination = WORK / 'sota_pd_et/predict.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/colab_download.py
embedded_source = r'''"""Download the exact official PADS v1.0.0 PD/ET input subset, using stdlib only.

The checksum list is pinned independently of the downloaded data. Every cached
or newly downloaded file must match that list. Only the participant table, two
preprocessing source files (for provenance), and 304 participants' JSON/movement
files are acquired. This module never downloads the complete dataset archive or
executes the official preprocessing scripts.
"""

from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
from datetime import datetime, timezone
import hashlib
from http.client import HTTPException
import json
import os
from pathlib import Path, PurePosixPath
import re
import tempfile
import time
from urllib.error import URLError
from urllib.parse import quote, urlsplit
from urllib.request import Request, urlopen


BASE_URL = "https://physionet.org/files/parkinsons-disease-smartwatch/1.0.0/"
CHECKSUM_LIST_SHA256 = "514cd95405a12afcdfb126d47d1f559e2e8a744f03e586c3088d5e4fd7b02c46"
EXPECTED_CONDITION_COUNTS = {"Parkinson's": 276, "Essential Tremor": 28}
EXPECTED_TABLE_SIZE = 469
MOVEMENT_BYTES = 132 * 976 * 4  # Published little-endian float32 representation.
TIMEOUT_SECONDS = 60
MAX_ATTEMPTS = 4
BLOCK_BYTES = 1024 * 1024


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(BLOCK_BYTES), b""):
            digest.update(block)
    return digest.hexdigest()


def _safe_relative(relative: str) -> str:
    """Accept only canonical relative POSIX paths, without traversal or URLs."""
    if not isinstance(relative, str) or not relative or "\\" in relative:
        raise ValueError(f"Unsafe dataset path: {relative!r}")
    path = PurePosixPath(relative)
    if (path.is_absolute() or any(p in {"", ".", ".."} for p in relative.split("/"))
            or ":" in relative or any(ord(c) < 32 for c in relative)
            or str(path) != relative):
        raise ValueError(f"Unsafe dataset path: {relative!r}")
    return relative


def _destination(root: Path, relative: str) -> Path:
    root = root.resolve()
    path = root / _safe_relative(relative)
    # Also reject existing symlinks that would direct writes outside the root.
    if not path.resolve().is_relative_to(root):
        raise ValueError(f"Dataset path escapes the output directory: {relative!r}")
    return path


def _read_checksums(path: Path) -> dict[str, str]:
    checksums = {}
    for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        fields = line.split(maxsplit=1)
        if len(fields) != 2 or re.fullmatch(r"[0-9a-fA-F]{64}", fields[0]) is None:
            raise ValueError(f"Malformed checksum-list line {number}.")
        relative = fields[1].removeprefix("*").removeprefix("./")
        relative = _safe_relative(relative)
        if relative in checksums:
            raise ValueError(f"Duplicate checksum-list path: {relative!r}")
        checksums[relative] = fields[0].lower()
    if not checksums:
        raise ValueError("The official checksum list is empty.")
    return checksums


def _download_verified(root: Path, relative: str, expected_sha256: str) -> dict:
    """Reuse a matching local file or atomically replace it after verification."""
    destination = _destination(root, relative)
    if destination.is_file() and sha256_file(destination) == expected_sha256:
        return {"path": relative, "sha256": expected_sha256,
                "bytes": destination.stat().st_size, "reused": True}
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = BASE_URL + quote(relative, safe="/")
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        temporary = None
        try:
            request = Request(url, headers={
                "User-Agent": "PADS-PD-ET-Colab/1.0 (selective research download)",
                "Accept-Encoding": "identity",
            })
            digest = hashlib.sha256()
            size = 0
            with urlopen(request, timeout=TIMEOUT_SECONDS) as response:
                resolved = urlsplit(response.geturl())
                origin = urlsplit(BASE_URL)
                if (resolved.scheme != "https" or resolved.netloc != origin.netloc
                        or not resolved.path.startswith(origin.path)):
                    raise ValueError(f"Unexpected download redirect: {response.geturl()}")
                with tempfile.NamedTemporaryFile(
                        mode="wb", dir=destination.parent,
                        prefix=f".{destination.name}.", suffix=".part", delete=False) as output:
                    temporary = Path(output.name)
                    while True:
                        block = response.read(BLOCK_BYTES)
                        if not block:
                            break
                        output.write(block)
                        digest.update(block)
                        size += len(block)
                    output.flush()
                    os.fsync(output.fileno())
            if digest.hexdigest() != expected_sha256:
                raise ValueError(f"SHA-256 mismatch for {relative}")
            os.replace(temporary, destination)
            return {"path": relative, "sha256": expected_sha256,
                    "bytes": size, "reused": False}
        except (OSError, URLError, HTTPException, ValueError) as error:
            last_error = error
            if attempt < MAX_ATTEMPTS:
                print(f"Retry {attempt}/{MAX_ATTEMPTS - 1}: {relative}: {error}", flush=True)
                time.sleep(min(2 ** (attempt - 1), 8))
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
    raise RuntimeError(
        f"Could not acquire verified file after {MAX_ATTEMPTS} attempts: {relative}"
    ) from last_error


def _select_participants(path: Path) -> list[dict]:
    with path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not {"id", "condition"}.issubset(reader.fieldnames):
            raise ValueError("The participant table needs id and condition columns.")
        rows = list(reader)
    ids = [row["id"] for row in rows]
    if any(not isinstance(pid, str) or re.fullmatch(r"[0-9]{3}", pid) is None for pid in ids):
        raise ValueError("Every participant ID must contain exactly three decimal digits.")
    if len(set(ids)) != len(ids):
        raise ValueError("Duplicate participant IDs in the official table.")
    if len(rows) != EXPECTED_TABLE_SIZE:
        raise ValueError(f"Expected {EXPECTED_TABLE_SIZE} official participants, found {len(rows)}.")
    selected = sorted((row for row in rows if row["condition"] in EXPECTED_CONDITION_COUNTS),
                      key=lambda row: row["id"])
    counts = {condition: sum(row["condition"] == condition for row in selected)
              for condition in EXPECTED_CONDITION_COUNTS}
    if counts != EXPECTED_CONDITION_COUNTS:
        raise ValueError(f"Unexpected PD/ET condition counts: {counts}")
    return selected


def download_dataset(root: str | Path, workers: int = 4) -> dict:
    """Acquire and validate the 304-participant PD/ET subset of PADS v1.0.0.

    ``root`` is the version directory (for example ``/content/pads/1.0.0``).
    Up to six workers are allowed to keep requests to PhysioNet modest. The
    returned JSON-serializable provenance dictionary can be saved by the caller
    alongside experiment results. No acquisition manifest is written into the
    source dataset directory. Re-running verifies hashes before reusing files.
    """
    if isinstance(workers, bool) or not isinstance(workers, int) or not 1 <= workers <= 6:
        raise ValueError("workers must be an integer between 1 and 6.")
    root = Path(root).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    started = datetime.now(timezone.utc).isoformat()
    print(f"Preparing verified PADS v1.0.0 PD/ET inputs in {root}", flush=True)
    results = [_download_verified(root, "SHA256SUMS.txt", CHECKSUM_LIST_SHA256)]
    checksums = _read_checksums(root / "SHA256SUMS.txt")
    table_name = "preprocessed/file_list.csv"
    if table_name not in checksums:
        raise ValueError(f"File missing from official checksum list: {table_name}")
    results.append(_download_verified(root, table_name, checksums[table_name]))
    participants = _select_participants(root / table_name)
    relative_files = ["scripts/run_preprocessing.py", "scripts/utils/data_handling.py"]
    for row in participants:
        relative_files.extend((f"patients/patient_{row['id']}.json",
                               f"preprocessed/movement/{row['id']}_ml.bin"))
    missing = [name for name in relative_files if name not in checksums]
    if missing:
        raise ValueError(f"Files missing from official checksum list: {missing}")
    print(f"Selected {len(participants)} participants: {EXPECTED_CONDITION_COUNTS}. "
          f"Checking {len(relative_files) + 2} files with {workers} workers.", flush=True)
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(_download_verified, root, name, checksums[name]): name
                   for name in relative_files}
        try:
            for completed, future in enumerate(as_completed(futures), 1):
                results.append(future.result())
                if completed % 25 == 0 or completed == len(relative_files):
                    reused = sum(result["reused"] for result in results)
                    print(f"Verified {completed + 2}/{len(relative_files) + 2} files "
                          f"({reused} reused).", flush=True)
        except BaseException:
            for future in futures:
                future.cancel()
            raise
    # Cross-check semantic identity as well as file hashes before model loading.
    for row in participants:
        pid = row["id"]
        patient_path = _destination(root, f"patients/patient_{pid}.json")
        patient = json.loads(patient_path.read_text(encoding="utf-8"))
        if patient.get("id") != pid or patient.get("condition") != row["condition"]:
            raise ValueError(f"Participant table/JSON disagreement: {pid}")
        movement_path = _destination(root, f"preprocessed/movement/{pid}_ml.bin")
        if movement_path.stat().st_size != MOVEMENT_BYTES:
            raise ValueError(f"Unexpected movement binary size for participant {pid}.")
    reused = sum(result["reused"] for result in results)
    metadata = {
        "dataset": "PADS v1.0.0", "dataset_doi": "https://doi.org/10.13026/m0w9-zx22",
        "base_url": BASE_URL, "data_root": str(root), "started_at_utc": started,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "checksum_list_sha256": CHECKSUM_LIST_SHA256,
        "participant_count": len(participants),
        "condition_counts": dict(EXPECTED_CONDITION_COUNTS),
        "participant_ids": [row["id"] for row in participants],
        "verified_file_count": len(results), "downloaded_files": len(results) - reused,
        "reused_files": reused, "verified_bytes": sum(result["bytes"] for result in results),
        "verified_source_sha256": {result["path"]: result["sha256"]
                                   for result in sorted(results, key=lambda item: item["path"])},
    }
    print(f"Ready: {len(participants)} participants; {len(results)} verified files; "
          f"{len(results) - reused} downloaded and {reused} reused.", flush=True)
    return metadata


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("root", type=Path, help="PADS version directory to populate")
    parser.add_argument("--workers", type=int, default=4, help="Concurrent requests (1-6)")
    arguments = parser.parse_args()
    record = download_dataset(arguments.root, arguments.workers)
    print(json.dumps({key: value for key, value in record.items()
                      if key not in {"participant_ids", "verified_source_sha256"}}, indent=2))
'''
destination = WORK / 'sota_pd_et/colab_download.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/colab_report.py
embedded_source = r'''"""Verify completed nested-CV outputs and create portable figures and a report.

This module never fits a model. Run with the same numerical environment as the
evaluation, plus matplotlib::

    python -m sota_pd_et.colab_report --run /path/to/results --output /path/to/report
"""

from __future__ import annotations

import argparse
import base64
from collections import Counter
import csv
import hashlib
import html
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve

from .evaluate import PERFORMANCE_METRICS, describe_values, metrics


CAVEATS = (
    "The positive class is Essential Tremor (ET=1); Parkinson's disease is PD=0. "
    "ET is predicted only when the ridge margin is strictly greater than zero.",
    "Reported performance uses outer held-out participant predictions. "
    "Inner alpha-selection results and the full-cohort final model's training predictions "
    "are not test performance.",
    "Decision scores are signed, uncalibrated ridge margins, not probabilities. "
    "Pooled AUROC and average precision compare margins from separately fitted models, "
    "whose numerical scales can differ; inspect the gray per-fold curves and fold metrics too.",
    "Fold means and standard deviations are descriptive, not confidence intervals, "
    "because training sets overlap. Repeating cross-validation does not create "
    "additional independent participants.",
    "This evaluation uses the published PADS movement representation, including its "
    "upstream preprocessing and task choices. It establishes neither an external "
    "clinical validation nor a state-of-the-art ranking.",
)
METRIC_LABELS = {
    "balanced_accuracy": "Balanced accuracy",
    "accuracy": "Accuracy",
    "roc_auc": "AUROC (ET positive)",
    "average_precision_et": "ET average precision",
    "et_recall": "ET recall",
    "pd_recall": "PD recall",
    "et_precision": "ET precision",
    "f1_et": "ET F1",
    "macro_f1": "Macro F1",
    "matthews_correlation": "Matthews correlation",
}


def require(condition, message):
    if not condition:
        raise ValueError(message)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_csv(path):
    with path.open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def compare_metrics(actual, saved, context):
    for key, value in actual.items():
        require(key in saved, f"{context}: missing metric {key}.")
        if isinstance(value, list):
            require(saved[key] == value, f"{context}: {key} disagrees.")
        else:
            expected = float(saved[key])
            require(np.isfinite(expected) and abs(expected - value) <= 1e-12,
                    f"{context}: {key} disagrees.")


def compare_descriptive(actual, saved, context):
    for key, value in actual.items():
        require(key in saved, f"{context}: missing metric {key}.")
        for statistic in ("mean", "std"):
            a, b = value[statistic], saved[key][statistic]
            require((a is None and b is None) or
                    (a is not None and b is not None and np.isfinite(float(b))
                     and abs(a - float(b)) <= 1e-12),
                    f"{context}: {key} {statistic} disagrees.")
        require(np.allclose(value["values"], saved[key]["values"], rtol=0, atol=1e-12),
                f"{context}: {key} values disagree.")


def verify_run(run):
    """Recompute metrics and verify participant coverage, folds, and stored labels."""
    run = Path(run).expanduser().resolve()
    names = ("manifest.json", "summary.json", "cohort.csv", "splits.json",
             "oof_predictions.csv", "fold_metrics.csv")
    manifest = read_json(run / "manifest.json")
    require(manifest.get("status") == "complete", "The evaluation manifest is not complete.")
    summary = read_json(run / "summary.json")
    config = manifest["config"]
    repeats, outer_folds = int(config["repeats"]), int(config["outer_folds"])
    require(repeats >= 1 and outer_folds >= 2, "Invalid repeat or outer fold count.")
    cohort = read_csv(run / "cohort.csv")
    targets = {row["participant_id"]: int(row["target_et"]) for row in cohort}
    require(len(targets) == len(cohort) > 0, "Cohort IDs must be unique and nonempty.")
    require(set(targets.values()) == {0, 1}, "Require exactly PD=0 and ET=1.")
    counts = Counter(targets.values())
    require(counts[0] == summary["cohort_counts"]["Parkinson's"] and
            counts[1] == summary["cohort_counts"]["Essential Tremor"],
            "Summary class counts disagree with the cohort.")
    predictions = read_csv(run / "oof_predictions.csv")
    require(len(predictions) == len(targets) * repeats,
            "Wrong total number of outer predictions.")
    for row in predictions:
        repeat, fold = int(row["repeat"]), int(row["fold"])
        require(1 <= repeat <= repeats and 1 <= fold <= outer_folds,
                "Prediction repeat/fold is outside the configured range.")
        pid, target, score = row["participant_id"], int(row["target_et"]), float(row["decision_score_et"])
        require(pid in targets and target == targets[pid], "Outer target disagrees with cohort.")
        require(np.isfinite(score), "Nonfinite outer decision score.")
        require(int(row["predicted_et"]) == int(score > 0.0),
                "Stored prediction disagrees with the fixed strict score>0 rule.")
    plans = read_json(run / "splits.json")
    expected_keys = {(r, f) for r in range(1, repeats + 1) for f in range(1, outer_folds + 1)}
    require(Counter((int(p["repeat"]), int(p["fold"])) for p in plans)
            == Counter({key: 1 for key in expected_keys}), "Outer split plan is incomplete or duplicated.")
    saved_folds = summary["outer_fold_metrics"]
    fold_csv = read_csv(run / "fold_metrics.csv")
    for rows, context in ((saved_folds, "Summary"), (fold_csv, "Metric CSV")):
        require(Counter((int(p["repeat"]), int(p["fold"])) for p in rows)
                == Counter({key: 1 for key in expected_keys}), f"{context}: incomplete or duplicate folds.")
    require(Counter(int(row["repeat"]) for row in summary["repeat_metrics"])
            == Counter({r: 1 for r in range(1, repeats + 1)}),
            "Summary repeat metrics are incomplete or duplicated.")
    repeat_metrics, fold_metrics, groups = [], [], {}
    for repeat in range(1, repeats + 1):
        group = [p for p in predictions if int(p["repeat"]) == repeat]
        require(Counter(p["participant_id"] for p in group)
                == Counter({pid: 1 for pid in targets}),
                f"Repeat {repeat}: each participant must be tested exactly once.")
        y, scores = arrays(group)
        result = {"repeat": repeat, **metrics(y, scores)}
        saved = next(p for p in summary["repeat_metrics"] if int(p["repeat"]) == repeat)
        compare_metrics(metrics(y, scores), saved, f"Repeat {repeat}")
        repeat_metrics.append(result)
        groups[repeat] = group
        for fold in range(1, outer_folds + 1):
            key = (repeat, fold)
            plan = next(p for p in plans if (int(p["repeat"]), int(p["fold"])) == key)
            train, test = set(plan["train_ids"]), set(plan["test_ids"])
            require(len(train) == len(plan["train_ids"]) and len(test) == len(plan["test_ids"])
                    and not train & test and train | test == set(targets),
                    f"Repeat {repeat}, fold {fold}: invalid participant split.")
            selected = [p for p in group if int(p["fold"]) == fold]
            require(set(p["participant_id"] for p in selected) == test,
                    f"Repeat {repeat}, fold {fold}: outer predictions disagree with split membership.")
            inner_valid = Counter()
            require(len(plan["inner_splits"]) == int(config["inner_folds"]),
                    f"Repeat {repeat}, fold {fold}: wrong inner fold count.")
            for inner in plan["inner_splits"]:
                a, b = set(inner["train_ids"]), set(inner["validation_ids"])
                require(len(a) == len(inner["train_ids"]) and len(b) == len(inner["validation_ids"])
                        and not a & b and a | b == train,
                        f"Repeat {repeat}, fold {fold}: invalid inner participant split.")
                inner_valid.update(inner["validation_ids"])
            require(inner_valid == Counter({pid: 1 for pid in train}),
                    f"Repeat {repeat}, fold {fold}: incomplete inner validation coverage.")
            computed = metrics(*arrays(selected))
            saved = next(p for p in saved_folds if (int(p["repeat"]), int(p["fold"])) == key)
            flat = next(p for p in fold_csv if (int(p["repeat"]), int(p["fold"])) == key)
            compare_metrics(computed, saved, f"Repeat {repeat}, fold {fold} summary")
            compare_metrics({k: v for k, v in computed.items() if not isinstance(v, list)}, flat,
                            f"Repeat {repeat}, fold {fold} CSV")
            require(float(saved["selected_alpha"]) == float(flat["selected_alpha"]),
                    f"Repeat {repeat}, fold {fold}: selected alpha disagrees.")
            fold_metrics.append({"repeat": repeat, "fold": fold,
                                 "selected_alpha": float(saved["selected_alpha"]), **computed})
    across_repeats = {key: describe_values([row[key] for row in repeat_metrics]) for key in PERFORMANCE_METRICS}
    across_folds = {key: describe_values([row[key] for row in fold_metrics]) for key in PERFORMANCE_METRICS}
    compare_descriptive(across_repeats, summary["across_repeats"], "Across repeats")
    compare_descriptive(across_folds, summary["across_outer_folds_descriptive"], "Across outer folds")
    y = np.array(list(targets.values()), dtype=int)
    baseline = metrics(y, -np.ones(len(y)))
    compare_metrics(baseline, summary["always_predict_pd_baseline"], "Always-PD baseline")
    require(abs(float(summary["average_precision_no_skill_reference"]) - float(y.mean())) <= 1e-12,
            "Average-precision reference disagrees with ET prevalence.")
    report = {
        "run_directory": str(run),
        "verification": "Passed: completed manifest; cohort labels/counts; one outer prediction per participant "
        "per repeat; fixed score>0 rule; outer/inner split membership and coverage; recomputed "
        "repeat/fold metrics and descriptive statistics; always-PD baseline.",
        "input_sha256": {name: hashlib.sha256((run / name).read_bytes()).hexdigest() for name in names},
        "cohort_counts": summary["cohort_counts"],
        "repeat_metrics": repeat_metrics,
        "outer_fold_metrics": fold_metrics,
        "across_repeats": across_repeats,
        "across_outer_folds_descriptive": across_folds,
        "always_predict_pd_baseline": baseline,
        "average_precision_no_skill_reference": float(y.mean()),
        "interpretation": list(CAVEATS),
    }
    return report, groups


def arrays(rows):
    return (np.array([int(p["target_et"]) for p in rows]),
            np.array([float(p["decision_score_et"]) for p in rows]))


def pr_steps(y, scores):
    """Return coordinates whose rectangle area equals noninterpolated AP."""
    precision, recall, _ = precision_recall_curve(y, scores)
    rx, py = recall[::-1], precision[::-1]
    x, p = [float(rx[0])], [float(py[0])]
    for index in range(1, len(rx)):
        x.extend((float(rx[index - 1]), float(rx[index])))
        p.extend((float(py[index]), float(py[index])))
    return x, p


def create_figures(output, report, groups):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    figures = []
    for result in report["repeat_metrics"]:
        repeat = result["repeat"]
        group = groups[repeat]
        y, scores = arrays(group)
        prefix = f"repeat_{repeat:02d}"
        matrix = np.asarray(result["confusion_matrix_true_rows_predicted_columns_pd_et"])
        fig, ax = plt.subplots(figsize=(5.6, 4.8), constrained_layout=True)
        image = ax.imshow(matrix, cmap="Blues", vmin=0)
        for (row, column), count in np.ndenumerate(matrix):
            ax.text(column, row, str(count), ha="center", va="center", fontsize=20,
                    color="white" if count > matrix.max() / 2 else "#17324d")
        ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["PD", "ET"], yticklabels=["PD", "ET"],
               xlabel="Predicted condition", ylabel="True condition",
               title=f"Outer held-out predictions — repeat {repeat}\n"
                     f"Balanced accuracy {result['balanced_accuracy']:.3f}")
        fig.colorbar(image, ax=ax, label="Participants")
        filename = f"{prefix}_confusion.png"
        fig.savefig(output / filename, dpi=170)
        plt.close(fig)
        figures.append(filename)
        for kind in ("roc", "pr"):
            fig, ax = plt.subplots(figsize=(6.2, 5.0), constrained_layout=True)
            fold_ids = sorted({int(p["fold"]) for p in group})
            for index, fold in enumerate(fold_ids):
                fy, fs = arrays([p for p in group if int(p["fold"]) == fold])
                x, value = roc_curve(fy, fs)[:2] if kind == "roc" else pr_steps(fy, fs)
                ax.plot(x, value, color="#a6adb6", linewidth=1.1, alpha=0.8,
                        label="Individual outer folds" if index == 0 else None)
            if kind == "roc":
                x, value, _ = roc_curve(y, scores)
                ax.plot([0, 1], [0, 1], color="#7e8792", linestyle="--", linewidth=1,
                        label="No discrimination")
                label = f"Pooled outer predictions (AUROC={result['roc_auc']:.3f})"
                axis_labels = {"xlabel": "False positive rate (PD called ET)", "ylabel": "ET recall"}
            else:
                x, value = pr_steps(y, scores)
                prevalence = float(y.mean())
                ax.axhline(prevalence, color="#7e8792", linestyle="--", linewidth=1,
                           label=f"ET prevalence={prevalence:.3f}")
                label = f"Pooled outer predictions (AP={result['average_precision_et']:.3f})"
                axis_labels = {"xlabel": "ET recall", "ylabel": "ET precision"}
            ax.plot(x, value, color="#087f8c", linewidth=2.3, label=label)
            ax.set(**axis_labels, xlim=(0, 1), ylim=(0, 1.02),
                   title=f"Outer held-out {'ROC' if kind == 'roc' else 'precision–recall'} — repeat {repeat}")
            ax.grid(alpha=0.2)
            ax.legend(loc="lower right" if kind == "roc" else "upper right", fontsize=8)
            fig.text(0.5, -0.035, "Pooled curves compare margins from different fitted models.",
                     ha="center", fontsize=9, color="#56616f")
            filename = f"{prefix}_{kind}.png"
            fig.savefig(output / filename, dpi=170, bbox_inches="tight")
            plt.close(fig)
            figures.append(filename)
    return figures


def write_report(output, report, figures):
    rows = []
    for scope, source in (("pooled_outer_repeat", report["repeat_metrics"]),
                          ("outer_fold", report["outer_fold_metrics"])):
        for row in source:
            rows.append({"scope": scope, "repeat": row["repeat"], "fold": row.get("fold", ""),
                         "selected_alpha": row.get("selected_alpha", ""),
                         **{key: value for key, value in row.items()
                            if key not in {"repeat", "fold", "selected_alpha"} and not isinstance(value, list)}})
    with (output / "report_metrics.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    report["figures"] = figures
    (output / "report.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    esc = html.escape
    columns = "".join(f"<th>Repeat {row['repeat']}</th>" for row in report["repeat_metrics"])
    body = []
    for key in PERFORMANCE_METRICS:
        values = "".join(f"<td>{row[key]:.4f}</td>" for row in report["repeat_metrics"])
        stat = report["across_outer_folds_descriptive"][key]
        deviation = "n/a" if stat["std"] is None else f"{stat['std']:.4f}"
        baseline = report["always_predict_pd_baseline"][key]
        body.append(f"<tr><th>{esc(METRIC_LABELS[key])}</th>{values}"
                    f"<td>{stat['mean']:.4f} ± {deviation}</td><td>{baseline:.4f}</td></tr>")
    images = []
    for name in figures:
        data = base64.b64encode((output / name).read_bytes()).decode("ascii")
        images.append(f'<img alt="{esc(name)}" src="data:image/png;base64,{data}">')
    notes = "".join(f"<li>{esc(note)}</li>" for note in CAVEATS)
    fold_rows = "".join(
        f"<tr><td>{row['repeat']}</td><td>{row['fold']}</td><td>{row['selected_alpha']:g}</td>"
        f"<td>{row['balanced_accuracy']:.4f}</td><td>{row['et_recall']:.4f}</td>"
        f"<td>{row['pd_recall']:.4f}</td><td>{row['roc_auc']:.4f}</td>"
        f"<td>{row['average_precision_et']:.4f}</td><td>{row['n_et']}</td></tr>"
        for row in report["outer_fold_metrics"]
    )
    counts = report["cohort_counts"]
    text = f"""<!doctype html>
<html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>PD/ET participant-level nested-CV report</title>
<style>body{{font:16px/1.55 system-ui,sans-serif;max-width:1200px;margin:32px auto;padding:0 20px;color:#17324d}}
table{{border-collapse:collapse;display:block;overflow:auto}}td,th{{padding:9px 14px;border:1px solid #dce3e8;text-align:right}}
th:first-child{{text-align:left}}thead{{background:#edf5f6}}.figures{{display:flex;flex-wrap:wrap;gap:12px}}
img{{width:min(100%,560px);height:auto;object-fit:contain}}code{{overflow-wrap:anywhere}}</style>
<h1>PD/ET participant-level nested cross-validation</h1>
<p>Participants: PD={counts["Parkinson's"]}; ET={counts['Essential Tremor']}.</p>
<p><strong>Verification passed.</strong> {esc(report['verification'])}</p>
<table><thead><tr><th>Metric</th>{columns}<th>Outer folds, descriptive mean ± SD</th><th>Always-PD baseline</th></tr></thead>
<tbody>{''.join(body)}</tbody></table>
<p>Repeat columns pool one held-out prediction per participant in each repeat. The fold column describes all outer folds across all repeats; SD is not a confidence interval.</p>
<h2>Individual outer folds</h2>
<table><thead><tr><th>Repeat</th><th>Fold</th><th>Selected alpha</th><th>Balanced accuracy</th><th>ET recall</th><th>PD recall</th><th>AUROC</th><th>ET AP</th><th>ET participants</th></tr></thead>
<tbody>{fold_rows}</tbody></table>
<h2>Interpretation</h2><ul>{notes}</ul>
<h2>Outer held-out predictions</h2><div class="figures">{''.join(images)}</div>
<p>Verified run: <code>{esc(report['run_directory'])}</code>. This report contains its figures and requires no external image files.</p>
</html>
"""
    (output / "report.html").write_text(text, encoding="utf-8")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run", type=Path, required=True)
    parser.add_argument("--output", type=Path, help="New report directory; default: RUN/colab_report.")
    args = parser.parse_args(argv)
    report, groups = verify_run(args.run)
    output = (args.output or args.run / "colab_report").expanduser().resolve()
    output.mkdir(parents=True, exist_ok=False)
    figures = create_figures(output, report, groups)
    write_report(output, report, figures)
    print(f"Verified outer evaluation and saved report: {output}")
    for result in report["repeat_metrics"]:
        print(f"Repeat {result['repeat']}: BA={result['balanced_accuracy']:.4f}; "
              f"AUROC={result['roc_auc']:.4f}; ET recall={result['et_recall']:.4f}; "
              f"PD recall={result['pd_recall']:.4f}; AP={result['average_precision_et']:.4f}")
    return output


if __name__ == "__main__":
    main()
'''
destination = WORK / 'sota_pd_et/colab_report.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/hydra_gpu.py
embedded_source = r'''"""Optional CUDA execution of the already fitted aeon 1.6.0 HYDRA map.

aeon's HYDRA weights and channel indices are ordinary tensor attributes, not
registered module buffers, and its forward method creates CPU accumulators.
Consequently, calling ``module.to('cuda')`` is insufficient.  This adapter
copies the *fitted* weights and indices to the selected device and places its
accumulators there too.  It never fits a map or draws random numbers.

Inputs and convolutions use float32, with TF32 disabled for CUDA convolutions.
CUDA reductions (in particular floating-point scatter additions) can still
produce small numerical differences from CPU execution; bitwise agreement and
identical predictions at a decision boundary are not guaranteed.

The forward computation is adapted from aeon 1.6.0's
``aeon.transformations.collection.convolution_based._hydra._HydraInternal``.
The original implementation is distributed under the following license:

BSD 3-Clause License

Copyright (c) The aeon developers.
Copyright (c) 2022 The sktime developers.

All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

* Neither the name of the copyright holder nor the names of its
  contributors may be used to endorse or promote products derived from
  this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
"""

from __future__ import annotations

from contextlib import nullcontext

import numpy as np


def resolve_device(requested: str = "auto") -> str:
    """Resolve ``auto``, ``cpu``, or ``cuda`` to a usable device name.

    ``auto`` chooses CUDA when it is available and CPU otherwise.  Explicit
    ``cuda`` requests fail clearly when PyTorch cannot use CUDA; they never
    silently switch to CPU.  Apple MPS is not a supported backend here.
    """
    import torch

    if requested not in ("auto", "cpu", "cuda"):
        raise ValueError("device must be one of 'auto', 'cpu', or 'cuda'.")
    if requested == "cpu":
        return "cpu"
    cuda_available = torch.cuda.is_available()
    if requested == "cuda" and not cuda_available:
        raise RuntimeError(
            "CUDA was requested but PyTorch cannot use an NVIDIA GPU. "
            "Select a GPU runtime in Colab, or choose device='cpu' or 'auto'."
        )
    return "cuda" if cuda_available else "cpu"


class HydraDeviceTransform:
    """A device cache for one fitted aeon ``HydraTransformer``.

    Parameters
    ----------
    fitted_transformer
        A fitted aeon 1.6.0 ``HydraTransformer``.  Its CPU tensors remain the
        authoritative portable model state and are never modified.
    device
        ``'auto'``, ``'cpu'``, or ``'cuda'``.  CPU is useful for verification
        against aeon's reference implementation.

    This object is an execution cache.  The owning feature map should omit it
    from serialization and recreate it for the destination machine.  All
    outputs are moved back to CPU before they are returned so the existing
    training-fold scaler and ridge classifier retain their CPU behavior.
    """

    def __init__(self, fitted_transformer, device: str = "cuda"):
        import torch

        self.device = resolve_device(device)
        internal = getattr(fitted_transformer, "_hydra", None)
        if internal is None:
            raise ValueError("Fit the aeon HydraTransformer before creating its device cache.")
        self.k = int(internal.k)
        self.h = int(internal.h)
        self.divisor = int(internal.divisor)
        self.dilations = tuple(int(value.item()) for value in internal.dilations)
        self.paddings = tuple(int(value.item()) for value in internal.paddings)
        # Copy existing values, preserving the exact fitted kernels and channel
        # selection.  In particular, do not instantiate another HYDRA module.
        self.weights = internal.W.detach().to(
            device=self.device, dtype=torch.float32, copy=True
        ).contiguous()
        self.indices = tuple(
            value.detach().to(device=self.device, dtype=torch.long, copy=True)
            for value in internal.idx
        )

    def transform(self, X: np.ndarray):
        """Return one batch's unscaled HYDRA features as a CPU float32 tensor.

        ``X`` has shape ``(participants, channels, timepoints)``.  Batch size
        is controlled by the caller; a CUDA allocation failure is allowed to
        propagate so an explicit GPU request does not silently change device.
        """
        import torch
        import torch.nn.functional as F

        array = np.asarray(X, dtype=np.float32)
        if array.ndim != 3 or min(array.shape) < 1:
            raise ValueError("Expected a nonempty (participants, channels, timepoints) batch.")
        # A copy also handles a read-only array or a view with negative strides.
        tensor = torch.from_numpy(np.array(array, dtype=np.float32, order="C", copy=True))
        tensor = tensor.to(device=self.device, dtype=torch.float32)
        n_examples, n_channels, _ = tensor.shape

        # Deterministic convolution selection and full float32 arithmetic avoid
        # the additional TF32 approximation.  Scatter sums can still differ in
        # their floating-point addition order on CUDA.
        convolution_context = (
            torch.backends.cudnn.flags(
                enabled=True, benchmark=False, deterministic=True, allow_tf32=False
            )
            if self.device == "cuda"
            else nullcontext()
        )
        with torch.inference_mode(), convolution_context:
            differenced = torch.diff(tensor) if self.divisor > 1 else None
            features = []
            for dilation_index, (dilation, padding) in enumerate(
                zip(self.dilations, self.paddings)
            ):
                for diff_index in range(self.divisor):
                    series = tensor if diff_index == 0 else differenced
                    weights = self.weights[dilation_index, diff_index]
                    if n_channels > 1:
                        selected = series[:, self.indices[dilation_index][diff_index]].sum(2)
                        responses = F.conv1d(
                            selected, weights, dilation=dilation,
                            padding=padding, groups=self.h,
                        )
                    else:
                        responses = F.conv1d(
                            series, weights, dilation=dilation, padding=padding,
                        )
                    responses = responses.view(n_examples, self.h, self.k, -1)
                    max_values, max_indices = responses.max(2)
                    min_values, min_indices = responses.min(2)
                    count_max = responses.new_zeros(n_examples, self.h, self.k)
                    count_min = responses.new_zeros(n_examples, self.h, self.k)
                    count_max.scatter_add_(-1, max_indices, max_values)
                    count_min.scatter_add_(-1, min_indices, torch.ones_like(min_values))
                    features.extend((count_max, count_min))
            return torch.cat(features, 1).view(n_examples, -1).cpu()
'''
destination = WORK / 'sota_pd_et/hydra_gpu.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/tests/__init__.py
embedded_source = r'''"""Focused data-integrity and evaluation-protocol tests."""
'''
destination = WORK / 'sota_pd_et/tests/__init__.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/tests/test_pipeline.py
embedded_source = r'''"""Tests use small synthetic fixtures, never train on the real PADS cohort."""

from __future__ import annotations

import csv
from importlib.util import find_spec
import json
from pathlib import Path
import tempfile
import unittest
from unittest import mock

import numpy as np

from sota_pd_et.data import CHANNEL_NAMES, MOVEMENT_SHAPE, load_cohort, sha256_file


HAS_SKLEARN = all(find_spec(p) is not None for p in ("sklearn", "joblib", "threadpoolctl"))
HAS_MODEL_DEPS = HAS_SKLEARN and all(find_spec(p) is not None for p in ("aeon", "torch"))


def make_fixture(root):
    (root / "preprocessed/movement").mkdir(parents=True)
    (root / "patients").mkdir()
    (root / "scripts/utils").mkdir(parents=True)
    (root / "scripts/run_preprocessing.py").write_text("# synthetic fixture\n")
    (root / "scripts/utils/data_handling.py").write_text("# synthetic fixture\n")
    rows = [
        {"id": "004", "condition": "Parkinson's", "label": "1", "disease_comment": "essential tremor history"},
        {"id": "066", "condition": "Essential Tremor", "label": "2", "disease_comment": "-"},
        {"id": "002", "condition": "Other Movement Disorders", "label": "2", "disease_comment": "-"},
    ]
    with (root / "preprocessed/file_list.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    for row in rows[:2]:
        (root / f"patients/patient_{row['id']}.json").write_text(json.dumps(row))
        np.full(MOVEMENT_SHAPE, 0.25, dtype="<f4").tofile(
            root / f"preprocessed/movement/{row['id']}_ml.bin")
    update_checksums(root)


def update_checksums(root):
    files = sorted(p for p in root.rglob("*") if p.is_file() and p.name != "SHA256SUMS.txt")
    (root / "SHA256SUMS.txt").write_text("".join(
        f"{sha256_file(p)}  {p.relative_to(root).as_posix()}\n" for p in files
    ))


class DataTests(unittest.TestCase):
    def setUp(self):
        self.temporary = tempfile.TemporaryDirectory()
        self.addCleanup(self.temporary.cleanup)
        self.root = Path(self.temporary.name)
        make_fixture(self.root)

    def test_condition_strings_define_binary_target(self):
        cohort = load_cohort(self.root)
        self.assertEqual(cohort.ids.tolist(), ["004", "066"])
        self.assertEqual(cohort.y.tolist(), [0, 1])
        self.assertEqual(cohort.X.shape, (2, 132, 976))
        self.assertEqual(cohort.provenance["pd_with_et_comment_ids"], ["004"])
        self.assertEqual(len(CHANNEL_NAMES), 132)
        self.assertEqual(CHANNEL_NAMES[0], "Relaxed1_LeftWrist_Accelerometer_X")
        self.assertEqual(CHANNEL_NAMES[-1], "Entrainment2_RightWrist_Gyroscope_Z")

    def test_checksum_tampering_is_rejected(self):
        path = self.root / "preprocessed/movement/066_ml.bin"
        with path.open("r+b") as stream:
            stream.write(b"bad!")
        with self.assertRaisesRegex(ValueError, "SHA-256 mismatch"):
            load_cohort(self.root)

    def test_nonfinite_samples_are_rejected_after_valid_checksum(self):
        path = self.root / "preprocessed/movement/066_ml.bin"
        values = np.fromfile(path, dtype="<f4")
        values[0] = np.nan
        values.tofile(path)
        update_checksums(self.root)
        with self.assertRaisesRegex(ValueError, "Nonfinite movement samples"):
            load_cohort(self.root)

    def test_patient_table_disagreement_is_rejected(self):
        path = self.root / "patients/patient_066.json"
        row = json.loads(path.read_text())
        row["condition"] = "Parkinson's"
        path.write_text(json.dumps(row))
        update_checksums(self.root)
        with self.assertRaisesRegex(ValueError, "table/JSON disagreement"):
            load_cohort(self.root)


@unittest.skipUnless(HAS_SKLEARN, "Install sota_pd_et/requirements.txt for protocol tests.")
class ProtocolTests(unittest.TestCase):
    def test_every_person_has_one_test_prediction_per_repeat_and_nested_splits(self):
        from sota_pd_et.evaluate import make_plan

        y = np.r_[np.zeros(80, dtype=int), np.ones(8, dtype=int)]
        ids = np.array([f"{i:03d}" for i in range(len(y))])
        plan = make_plan(y, ids, outer_folds=4, inner_folds=3, repeats=2, seed=17)
        self.assertEqual(plan, make_plan(y, ids, 4, 3, 2, 17))
        for repeat in (1, 2):
            test_ids = [p for s in plan if s["repeat"] == repeat for p in s["test_ids"]]
            self.assertCountEqual(test_ids, ids.tolist())
        for split in plan:
            train, test = set(split["train_ids"]), set(split["test_ids"])
            self.assertFalse(train & test)
            inner_valid = []
            for inner in split["inner_splits"]:
                a, b = set(inner["train_ids"]), set(inner["validation_ids"])
                self.assertFalse(a & b)
                self.assertEqual(a | b, train)
                self.assertFalse((a | b) & test)
                inner_valid.extend(inner["validation_ids"])
            self.assertCountEqual(inner_valid, split["train_ids"])

    def test_impossible_inner_stratification_is_rejected(self):
        from sota_pd_et.evaluate import make_plan

        y = np.r_[np.zeros(20, dtype=int), np.ones(4, dtype=int)]
        with self.assertRaisesRegex(ValueError, "Inner fold count"):
            make_plan(y, np.arange(len(y)), 4, 4, 1, 17)

    def test_metric_orientation_and_fixed_zero_threshold(self):
        from sota_pd_et.evaluate import metrics

        result = metrics(np.array([0, 0, 1, 1]), np.array([-2.0, 0.0, 0.0, 2.0]))
        self.assertEqual(result["confusion_matrix_true_rows_predicted_columns_pd_et"], [[2, 0], [1, 1]])
        self.assertAlmostEqual(result["balanced_accuracy"], 0.75)
        self.assertAlmostEqual(result["et_recall"], 0.5)
        self.assertAlmostEqual(result["pd_recall"], 1.0)
        self.assertAlmostEqual(result["roc_auc"], 0.875)

    def test_ties_choose_stronger_regularization(self):
        from sota_pd_et.evaluate import choose_alpha

        rows = [{"alpha": a, "inner_mean_balanced_accuracy": score}
                for a, score in [(0.1, 0.8), (1.0, 0.7), (10.0, 0.8)]]
        self.assertEqual(choose_alpha(rows), 10.0)

    def test_feature_maps_are_refitted_without_inner_validation_people(self):
        import sota_pd_et.evaluate as evaluation
        from sota_pd_et.model import ModelConfig

        X = np.zeros((16, 1, 16), dtype=np.float32)
        X[:, 0, 0] = np.arange(16)
        y = np.tile([0, 1], 8)
        fitted_ids = []
        test_case = self

        class TrackingMap:
            def __init__(self, config, seed):
                pass

            def fit_transform(self, values):
                self.people = set(values[:, 0, 0].tolist())
                fitted_ids.append(self.people)
                return values[:, :, 0]

            def transform(self, values):
                test_case.assertFalse(self.people & set(values[:, 0, 0].tolist()))
                return values[:, :, 0]

        class FixedMargin:
            def decision_function(self, features):
                return 2 * (features[:, 0] % 2) - 1

        inner = evaluation.make_inner_splits(y, 4, 17)
        with mock.patch.object(evaluation, "FeatureMap", TrackingMap), \
                mock.patch.object(evaluation, "fit_ridge", return_value=FixedMargin()):
            alpha, _, scores = evaluation.tune_alpha(X, y, ModelConfig(), [0.1, 10.0], inner, 17)
        self.assertEqual(len(fitted_ids), 4)
        self.assertTrue(all(len(people) == 12 for people in fitted_ids))
        self.assertEqual(alpha, 10.0)
        np.testing.assert_array_equal(scores > 0, y)

    def test_ridge_score_orientation(self):
        from sota_pd_et.model import fit_ridge

        Z = np.array([[-3.0, 0.0], [-2.0, 1.0], [2.0, -1.0], [3.0, 0.0]])
        y = np.array([0, 0, 1, 1])
        classifier = fit_ridge(Z, y, 1.0)
        np.testing.assert_array_equal(classifier.decision_function(Z) > 0, y)


@unittest.skipUnless(HAS_MODEL_DEPS, "Install full dependencies for the synthetic model/serialization test.")
class ModelSmokeTests(unittest.TestCase):
    def test_batched_real_features_and_serialization(self):
        import joblib
        from sota_pd_et.evaluate import fit_full_model
        from sota_pd_et.model import ModelConfig

        rng = np.random.default_rng(7)
        X = rng.normal(size=(12, 3, 64)).astype(np.float32)
        y = np.tile([0, 1], 6)
        config = ModelConfig(mr_kernels=84, hydra_groups=4, hydra_kernels=2,
                             batch_size=3, n_jobs=1)
        fitted = fit_full_model(X[:8], y[:8], config, alpha=1.0, seed=17)
        before = fitted.decision_function(X[8:])
        self.assertEqual(before.shape, (4,))
        self.assertTrue(np.isfinite(before).all())
        with tempfile.TemporaryDirectory() as temporary:
            path = Path(temporary) / "model.joblib"
            joblib.dump(fitted, path)
            after = joblib.load(path).decision_function(X[8:])
        np.testing.assert_allclose(before, after, rtol=1e-12, atol=1e-12)

    def test_complete_synthetic_evaluation_and_prediction_entry_point(self):
        import contextlib
        import io
        import joblib
        import sota_pd_et.evaluate as evaluation
        from sota_pd_et.data import Cohort
        from sota_pd_et.predict import main as predict

        rng = np.random.default_rng(19)
        X = rng.normal(size=(16, 3, 64)).astype(np.float32)
        y = np.r_[np.zeros(12, dtype=int), np.ones(4, dtype=int)]
        ids = np.array([f"{i:03d}" for i in range(16)])
        cohort = Cohort(X, y, ids, [
            {"participant_id": pid, "condition": "Essential Tremor" if target else "Parkinson's",
             "target_et": int(target), "pd_with_et_comment": False}
            for pid, target in zip(ids, y, strict=True)
        ], {"condition_counts": {"Parkinson's": 12, "Essential Tremor": 4},
            "channel_names": ["synthetic_x", "synthetic_y", "synthetic_z"]})
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            output = root / "results"
            args = evaluation.parse_args([
                "--data-root", str(root / "unused_source"), "--output", str(output),
                "--outer-folds", "2", "--inner-folds", "2", "--repeats", "1",
                "--n-jobs", "1", "--batch-size", "3", "--mr-kernels", "84",
                "--hydra-groups", "4", "--hydra-kernels", "2", "--alphas", "0.1", "1",
                "--fit-final",
            ])
            with mock.patch.object(evaluation, "load_cohort", return_value=cohort), \
                    contextlib.redirect_stdout(io.StringIO()):
                evaluation.run(args)
            self.assertEqual(json.loads((output / "manifest.json").read_text())["status"], "complete")
            with (output / "oof_predictions.csv").open(newline="") as stream:
                predictions = list(csv.DictReader(stream))
            self.assertCountEqual([row["participant_id"] for row in predictions], ids.tolist())
            summary = json.loads((output / "summary.json").read_text())
            self.assertEqual(summary["repeat_metrics"][0]["n_pd"], 12)
            self.assertEqual(summary["repeat_metrics"][0]["n_et"], 4)
            self.assertAlmostEqual(summary["always_predict_pd_baseline"]["balanced_accuracy"], 0.5)
            np.save(root / "new.npy", X[:2])
            with contextlib.redirect_stdout(io.StringIO()):
                predict(["--model", str(output / "final_model.joblib"),
                         "--inputs", str(root / "new.npy"), "--output", str(root / "predictions.csv")])
            with (root / "predictions.csv").open(newline="") as stream:
                actual = [float(row["decision_score_et"]) for row in csv.DictReader(stream)]
            expected = joblib.load(output / "final_model.joblib")["model"].decision_function(X[:2])
            np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)


if __name__ == "__main__":
    unittest.main()
'''
destination = WORK / 'sota_pd_et/tests/test_pipeline.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/tests/test_colab_download.py
embedded_source = r'''"""Offline acquisition tests: synthetic HTTP responses, never real downloads."""

from __future__ import annotations

from contextlib import ExitStack
import csv
import hashlib
from http.client import IncompleteRead
import io
import json
from pathlib import Path
import tempfile
import unittest
from unittest import mock
from urllib.error import URLError
from urllib.parse import unquote, urlsplit

from sota_pd_et import colab_download as download


def digest(content):
    return hashlib.sha256(content).hexdigest()


class FakeResponse(io.BytesIO):
    def __init__(self, content, url):
        super().__init__(content)
        self.url = url

    def geturl(self):
        return self.url


class FakeHTTP:
    def __init__(self, files, outcomes=None):
        self.files = files
        self.outcomes = outcomes or {}
        self.calls = []

    def __call__(self, request, timeout):
        assert timeout == download.TIMEOUT_SECONDS
        name = unquote(urlsplit(request.full_url).path.removeprefix(urlsplit(download.BASE_URL).path))
        self.calls.append(name)
        queued = self.outcomes.get(name)
        outcome = queued.pop(0) if queued else self.files[name]
        if isinstance(outcome, BaseException):
            raise outcome
        return FakeResponse(outcome, request.full_url)


def fixture_files():
    text = io.StringIO(newline="")
    writer = csv.DictWriter(text, fieldnames=["id", "condition", "label"])
    writer.writeheader()
    rows = [
        {"id": "004", "condition": "Parkinson's", "label": "1"},
        {"id": "066", "condition": "Essential Tremor", "label": "2"},
        {"id": "002", "condition": "Other Movement Disorders", "label": "2"},
    ]
    writer.writerows(rows)
    files = {
        "preprocessed/file_list.csv": text.getvalue().encode(),
        "scripts/run_preprocessing.py": b"# provenance only\n",
        "scripts/utils/data_handling.py": b"# provenance only\n",
    }
    for row in rows[:2]:
        files[f"patients/patient_{row['id']}.json"] = json.dumps(row).encode()
        files[f"preprocessed/movement/{row['id']}_ml.bin"] = b"\x00" * 16
    files["SHA256SUMS.txt"] = "".join(
        f"{digest(content)}  {name}\n" for name, content in sorted(files.items())
    ).encode()
    return files


class DownloadTests(unittest.TestCase):
    def setUp(self):
        temporary = tempfile.TemporaryDirectory()
        self.addCleanup(temporary.cleanup)
        self.root = Path(temporary.name)
        self.files = fixture_files()
        self.stack = ExitStack()
        self.addCleanup(self.stack.close)
        self.stack.enter_context(mock.patch.object(download, "CHECKSUM_LIST_SHA256",
                                                  digest(self.files["SHA256SUMS.txt"])))
        self.stack.enter_context(mock.patch.object(download, "EXPECTED_CONDITION_COUNTS",
                                                  {"Parkinson's": 1, "Essential Tremor": 1}))
        self.stack.enter_context(mock.patch.object(download, "EXPECTED_TABLE_SIZE", 3))
        self.stack.enter_context(mock.patch.object(download, "MOVEMENT_BYTES", 16))
        self.stack.enter_context(mock.patch.object(download.time, "sleep"))
        self.stack.enter_context(mock.patch("builtins.print"))

    def test_only_selected_files_acquired_and_verified_cache_reused(self):
        http = FakeHTTP(self.files)
        with mock.patch.object(download, "urlopen", side_effect=http):
            first = download.download_dataset(self.root, workers=2)
            second = download.download_dataset(self.root, workers=2)
        self.assertEqual(set(http.calls), set(self.files))
        self.assertEqual(len(http.calls), 8)
        self.assertEqual(first["participant_ids"], ["004", "066"])
        self.assertEqual(first["downloaded_files"], 8)
        self.assertEqual(first["reused_files"], 0)
        self.assertEqual(second["downloaded_files"], 0)
        self.assertEqual(second["reused_files"], 8)
        self.assertFalse(list(self.root.rglob("*.part")))
        self.assertFalse((self.root / "download_manifest.json").exists())
        for name, content in self.files.items():
            self.assertEqual((self.root / name).read_bytes(), content)

    def test_network_errors_truncated_reads_and_checksum_corruption_are_retried(self):
        name = "scripts/run_preprocessing.py"
        http = FakeHTTP(self.files, outcomes={name: [
            URLError("offline test"), IncompleteRead(b"partial", 100), b"corruption",
            self.files[name],
        ]})
        with mock.patch.object(download, "urlopen", side_effect=http):
            result = download._download_verified(self.root, name, digest(self.files[name]))
        self.assertFalse(result["reused"])
        self.assertEqual(http.calls, [name] * 4)
        self.assertEqual((self.root / name).read_bytes(), self.files[name])
        self.assertFalse(list(self.root.rglob("*.part")))

    def test_failed_download_preserves_existing_file_and_cleans_temporary_files(self):
        name = "preprocessed/file_list.csv"
        target = self.root / name
        target.parent.mkdir()
        target.write_bytes(b"old invalid cache")
        http = FakeHTTP({name: b"bad hash"})
        with mock.patch.object(download, "urlopen", side_effect=http):
            with self.assertRaisesRegex(RuntimeError, "after 4 attempts"):
                download._download_verified(self.root, name, digest(self.files[name]))
        self.assertEqual(target.read_bytes(), b"old invalid cache")
        self.assertFalse(list(self.root.rglob("*.part")))

    def test_corrupted_cache_is_replaced(self):
        name = "scripts/run_preprocessing.py"
        target = self.root / name
        target.parent.mkdir()
        target.write_bytes(b"corrupted local copy")
        http = FakeHTTP(self.files)
        with mock.patch.object(download, "urlopen", side_effect=http):
            result = download._download_verified(self.root, name, digest(self.files[name]))
        self.assertFalse(result["reused"])
        self.assertEqual(http.calls, [name])
        self.assertEqual(target.read_bytes(), self.files[name])

    def test_checksum_pin_rejects_wrong_list_before_table_download(self):
        http = FakeHTTP({"SHA256SUMS.txt": b"untrusted list"})
        with mock.patch.object(download, "urlopen", side_effect=http):
            with self.assertRaisesRegex(RuntimeError, "SHA256SUMS"):
                download.download_dataset(self.root)
        self.assertEqual(http.calls, ["SHA256SUMS.txt"] * 4)
        self.assertFalse((self.root / "SHA256SUMS.txt").exists())

    def test_all_table_ids_validated_including_excluded_condition(self):
        name = self.root / "table.csv"
        name.write_text("id,condition\n004,Parkinson's\n066,Essential Tremor\n../x,Other\n")
        with self.assertRaisesRegex(ValueError, "exactly three"):
            download._select_participants(name)
        name.write_text("id,condition\n004,Parkinson's\n066,Essential Tremor\n066,Other\n")
        with self.assertRaisesRegex(ValueError, "Duplicate"):
            download._select_participants(name)

    def test_condition_strings_not_numeric_labels_define_selection(self):
        name = self.root / "table.csv"
        name.write_bytes(self.files["preprocessed/file_list.csv"])
        self.assertEqual([r["id"] for r in download._select_participants(name)], ["004", "066"])
        name.write_text("id,condition\n004,Parkinson's\n066,Other\n002,Other\n")
        with self.assertRaisesRegex(ValueError, "condition counts"):
            download._select_participants(name)

    def test_traversals_urls_and_external_symlinks_are_rejected(self):
        for name in ["../x", "/absolute", "x/../y", "x//y", "x\\y", "https://host/x", "./x"]:
            with self.subTest(name=name), self.assertRaises(ValueError):
                download._destination(self.root, name)
        with tempfile.TemporaryDirectory() as outside:
            (self.root / "escape").symlink_to(outside, target_is_directory=True)
            with self.assertRaisesRegex(ValueError, "escapes"):
                download._destination(self.root, "escape/x")

    def test_malformed_and_duplicate_checksum_records_rejected(self):
        checksum = self.root / "checksum.txt"
        for content in ["not-a-checksum x", f"{'0' * 64} ../escape\n",
                        f"{'0' * 64} x\n{'1' * 64} x\n"]:
            checksum.write_text(content)
            with self.subTest(content=content), self.assertRaises(ValueError):
                download._read_checksums(checksum)

    def test_worker_limit_is_enforced(self):
        for workers in [0, 7, True, 1.5, "4"]:
            with self.subTest(workers=workers), self.assertRaisesRegex(ValueError, "workers"):
                download.download_dataset(self.root, workers=workers)


if __name__ == "__main__":
    unittest.main()
'''
destination = WORK / 'sota_pd_et/tests/test_colab_download.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


In [ ]:
# Embedded file: sota_pd_et/tests/test_gpu.py
embedded_source = r'''"""Optional CUDA HYDRA tests; synthetic fixtures require no downloaded data."""

from __future__ import annotations

from importlib.util import find_spec
import io
import pickle
import unittest
from unittest import mock

import numpy as np


HAS_TORCH = find_spec("torch") is not None
HAS_MODEL_DEPS = HAS_TORCH and all(
    find_spec(name) is not None for name in ("aeon", "sklearn", "joblib")
)
if HAS_TORCH:
    import torch

    HAS_CUDA = torch.cuda.is_available()
else:
    HAS_CUDA = False


@unittest.skipUnless(HAS_TORCH, "Install torch for optional device-selection tests.")
class DeviceSelectionTests(unittest.TestCase):
    def test_cpu_is_accepted_without_cuda_hardware(self):
        from sota_pd_et.hydra_gpu import resolve_device

        with mock.patch("torch.cuda.is_available", return_value=False):
            self.assertEqual(resolve_device("cpu"), "cpu")

    def test_auto_selects_cuda_when_available(self):
        from sota_pd_et.hydra_gpu import resolve_device

        with mock.patch("torch.cuda.is_available", return_value=True):
            self.assertEqual(resolve_device("auto"), "cuda")
            self.assertEqual(resolve_device("cuda"), "cuda")

    def test_auto_falls_back_to_cpu_when_cuda_is_unavailable(self):
        from sota_pd_et.hydra_gpu import resolve_device

        with mock.patch("torch.cuda.is_available", return_value=False):
            self.assertEqual(resolve_device("auto"), "cpu")

    def test_explicit_cuda_fails_clearly_when_unavailable(self):
        from sota_pd_et.hydra_gpu import resolve_device

        with mock.patch("torch.cuda.is_available", return_value=False):
            with self.assertRaisesRegex(RuntimeError, "CUDA|cuda"):
                resolve_device("cuda")

    def test_unknown_device_is_rejected(self):
        from sota_pd_et.hydra_gpu import resolve_device

        with self.assertRaises(ValueError):
            resolve_device("gpu")


@unittest.skipUnless(HAS_MODEL_DEPS, "Install full dependencies for HYDRA parity tests.")
class HydraAdapterTests(unittest.TestCase):
    @staticmethod
    def fixture(n_channels, n_timepoints, n_groups=4):
        from aeon.transformations.collection.convolution_based import HydraTransformer

        rng = np.random.default_rng(71 + n_channels + n_timepoints)
        X = rng.normal(size=(5, n_channels, n_timepoints)).astype(np.float32)
        hydra = HydraTransformer(
            n_kernels=2, n_groups=n_groups, n_jobs=1,
            random_state=17, output_type="tensor",
        ).fit(X)
        return X, hydra

    def test_cpu_adapter_matches_upstream_values_and_column_order(self):
        from sota_pd_et.hydra_gpu import HydraDeviceTransform

        for n_channels in (1, 3):
            for n_timepoints in (64, 65):
                with self.subTest(channels=n_channels, timepoints=n_timepoints):
                    X, hydra = self.fixture(n_channels, n_timepoints)
                    expected = hydra.transform(X)
                    actual = HydraDeviceTransform(hydra, "cpu").transform(X)
                    self.assertIsInstance(actual, torch.Tensor)
                    self.assertEqual(actual.device.type, "cpu")
                    self.assertFalse(actual.requires_grad)
                    torch.testing.assert_close(actual, expected, rtol=0, atol=0)

    def test_one_group_without_difference_branch_matches_upstream(self):
        from sota_pd_et.hydra_gpu import HydraDeviceTransform

        for n_channels in (1, 3):
            with self.subTest(channels=n_channels):
                X, hydra = self.fixture(n_channels, 65, n_groups=1)
                actual = HydraDeviceTransform(hydra, "cpu").transform(X)
                torch.testing.assert_close(actual, hydra.transform(X), rtol=0, atol=0)

    def test_adapter_preserves_inputs_and_upstream_cpu_tensors(self):
        from sota_pd_et.hydra_gpu import HydraDeviceTransform

        X, hydra = self.fixture(3, 64)
        original = X.copy()
        weights = hydra._hydra.W.clone()
        indices = [index.clone() for index in hydra._hydra.idx]
        adapter = HydraDeviceTransform(hydra, "cpu")
        first = adapter.transform(X)
        second = adapter.transform(X)
        np.testing.assert_array_equal(X, original)
        torch.testing.assert_close(first, second, rtol=0, atol=0)
        torch.testing.assert_close(hydra._hydra.W, weights, rtol=0, atol=0)
        self.assertEqual(hydra._hydra.W.device.type, "cpu")
        for actual, expected in zip(hydra._hydra.idx, indices, strict=True):
            self.assertEqual(actual.device.type, "cpu")
            torch.testing.assert_close(actual, expected, rtol=0, atol=0)

    @unittest.skipUnless(HAS_CUDA, "A working CUDA GPU is required for numerical CUDA verification.")
    def test_cuda_adapter_returns_cpu_features_close_to_upstream(self):
        from sota_pd_et.hydra_gpu import HydraDeviceTransform

        for n_channels in (1, 3):
            for n_timepoints in (64, 65):
                with self.subTest(channels=n_channels, timepoints=n_timepoints):
                    X, hydra = self.fixture(n_channels, n_timepoints)
                    expected = hydra.transform(X)
                    actual = HydraDeviceTransform(hydra, "cuda").transform(X)
                    self.assertEqual(actual.device.type, "cpu")
                    self.assertFalse(actual.requires_grad)
                    self.assertTrue(torch.isfinite(actual).all().item())
                    # HYDRA contains winner selection, so CUDA convolution rounding
                    # need not give bit-identical features. Random fixtures avoid
                    # deliberate ties while checking translation and feature order.
                    torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-3)
                    self.assertEqual(hydra._hydra.W.device.type, "cpu")
                    self.assertTrue(all(index.device.type == "cpu" for index in hydra._hydra.idx))


@unittest.skipUnless(HAS_MODEL_DEPS, "Install full dependencies for model portability tests.")
class ModelDevicePortabilityTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        from sota_pd_et.evaluate import fit_full_model
        from sota_pd_et.model import ModelConfig

        rng = np.random.default_rng(37)
        X = rng.normal(size=(12, 3, 64)).astype(np.float32)
        y = np.tile([0, 1], 6)
        config = ModelConfig(
            mr_kernels=84, hydra_groups=4, hydra_kernels=2,
            batch_size=3, n_jobs=1, hydra_device="cpu",
        )
        cls.fitted = fit_full_model(X[:8], y[:8], config, alpha=1.0, seed=17)
        cls.test_inputs = X[8:]

    def copy_model(self):
        return pickle.loads(pickle.dumps(self.fitted))

    def test_cpu_default_and_explicit_cpu_preserve_predictions(self):
        from sota_pd_et.model import ModelConfig

        self.assertEqual(ModelConfig().hydra_device, "cpu")
        fitted = self.copy_model()
        expected = fitted.decision_function(self.test_inputs)
        fitted.features.set_hydra_device("cpu")
        actual = fitted.decision_function(self.test_inputs)
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)

    def test_auto_cpu_fallback_preserves_predictions_after_serialization(self):
        fitted = self.copy_model()
        expected = fitted.decision_function(self.test_inputs)
        with mock.patch("torch.cuda.is_available", return_value=False):
            fitted.features.set_hydra_device("auto")
            loaded = pickle.loads(pickle.dumps(fitted))
            actual = loaded.decision_function(self.test_inputs)
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)
        self.assertEqual(loaded.features.hydra_device_, "cpu")

    def test_legacy_saved_cpu_model_without_device_attributes_still_predicts(self):
        fitted = self.copy_model()
        expected = fitted.decision_function(self.test_inputs)
        fitted.features.__dict__.pop("_hydra_device_request_", None)
        fitted.features.__dict__.pop("hydra_device_", None)
        fitted.features.__dict__.pop("_hydra_device_transform_", None)
        # Pickles made before this change have no new dataclass instance field.
        fitted.features.config.__dict__.pop("hydra_device", None)
        loaded = pickle.loads(pickle.dumps(fitted))
        with mock.patch("torch.cuda.is_available", return_value=False):
            actual = loaded.decision_function(self.test_inputs)
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)

    def test_failed_device_override_keeps_previous_working_selection(self):
        fitted = self.copy_model()
        expected = fitted.decision_function(self.test_inputs)
        with mock.patch("torch.cuda.is_available", return_value=False):
            with self.assertRaises(RuntimeError):
                fitted.features.set_hydra_device("cuda")
            actual = fitted.decision_function(self.test_inputs)
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)

    def test_gpu_backend_selection_and_cache_with_a_cpu_execution_double(self):
        from sota_pd_et.hydra_gpu import HydraDeviceTransform

        fitted = self.copy_model()
        expected = fitted.decision_function(self.test_inputs)

        def cpu_execution_double(transformer, device):
            self.assertEqual(device, "cuda")
            return HydraDeviceTransform(transformer, "cpu")

        # Exercise FeatureMap's GPU branch and cache on CPU test hosts. The
        # separate hardware-conditional tests provide actual GPU verification.
        with mock.patch("torch.cuda.is_available", return_value=True), \
                mock.patch("sota_pd_et.hydra_gpu.HydraDeviceTransform",
                           side_effect=cpu_execution_double) as factory:
            fitted.features.set_hydra_device("auto")
            first = fitted.decision_function(self.test_inputs)
            second = fitted.decision_function(self.test_inputs)
            self.assertEqual(fitted.features.hydra_device_, "cuda")
        self.assertEqual(factory.call_count, 1)
        np.testing.assert_allclose(first, expected, rtol=1e-12, atol=1e-12)
        np.testing.assert_allclose(second, expected, rtol=1e-12, atol=1e-12)

    def test_adapter_cache_is_excluded_from_saved_models(self):
        import joblib

        fitted = self.copy_model()
        expected = fitted.decision_function(self.test_inputs)

        class UnpicklableAdapter:
            def __reduce__(self):
                raise AssertionError("The device adapter cache must not be serialized.")

        adapter = UnpicklableAdapter()
        fitted.features._hydra_device_transform_ = adapter
        stream = io.BytesIO()
        joblib.dump(fitted, stream)
        self.assertIs(fitted.features._hydra_device_transform_, adapter)
        stream.seek(0)
        loaded = joblib.load(stream)
        self.assertNotIn("_hydra_device_transform_", loaded.features.__dict__)
        actual = loaded.decision_function(self.test_inputs)
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)
        self.assertEqual(loaded.features.hydra_._hydra.W.device.type, "cpu")
        self.assertTrue(all(
            index.device.type == "cpu" for index in loaded.features.hydra_._hydra.idx
        ))

    def test_saved_cuda_policy_falls_back_on_a_cpu_only_host(self):
        fitted = self.copy_model()
        expected = fitted.decision_function(self.test_inputs)
        # Simulate the saved selection without constructing CUDA tensors. The
        # saved fitted kernels and scalers remain CPU resident on a GPU host too.
        fitted.features._hydra_device_request_ = "cuda"
        fitted.features.hydra_device_ = "cuda"
        stream = pickle.dumps(fitted)
        with mock.patch("torch.cuda.is_available", return_value=False):
            loaded = pickle.loads(stream)
            actual = loaded.decision_function(self.test_inputs)
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)
        self.assertEqual(loaded.features.hydra_device_, "cpu")

    @unittest.skipUnless(HAS_CUDA, "A working CUDA GPU is required for model integration verification.")
    def test_real_cuda_training_and_saved_model_cpu_inference(self):
        import joblib
        from sota_pd_et.evaluate import fit_full_model
        from sota_pd_et.model import ModelConfig

        rng = np.random.default_rng(43)
        X = rng.normal(size=(12, 3, 65)).astype(np.float32)
        y = np.tile([0, 1], 6)
        config = ModelConfig(
            mr_kernels=84, hydra_groups=4, hydra_kernels=2,
            batch_size=3, n_jobs=1, hydra_device="cuda",
        )
        fitted = fit_full_model(X[:8], y[:8], config, alpha=1.0, seed=17)
        features = fitted.features
        self.assertEqual(features.hydra_device_, "cuda")
        adapter = features._hydra_device_transform_
        self.assertEqual(adapter.device, "cuda")
        self.assertEqual(adapter.weights.device.type, "cuda")
        self.assertTrue(all(index.device.type == "cuda" for index in adapter.indices))
        gpu_scores = fitted.decision_function(X[8:])
        self.assertEqual(gpu_scores.shape, (4,))
        self.assertTrue(np.isfinite(gpu_scores).all())
        self.assertEqual(features.hydra_._hydra.W.device.type, "cpu")
        self.assertTrue(all(index.device.type == "cpu" for index in features.hydra_._hydra.idx))
        for name in ("epsilon", "mu", "sigma"):
            self.assertEqual(getattr(features.hydra_scaler_, name).device.type, "cpu")

        stream = io.BytesIO()
        joblib.dump(fitted, stream)
        self.assertIs(features._hydra_device_transform_, adapter)
        stream.seek(0)
        loaded = joblib.load(stream)
        self.assertNotIn("_hydra_device_transform_", loaded.features.__dict__)
        self.assertEqual(loaded.features.hydra_._hydra.W.device.type, "cpu")
        self.assertTrue(all(
            index.device.type == "cpu" for index in loaded.features.hydra_._hydra.idx
        ))
        for name in ("epsilon", "mu", "sigma"):
            self.assertEqual(getattr(loaded.features.hydra_scaler_, name).device.type, "cpu")
        with mock.patch("torch.cuda.is_available", return_value=False):
            cpu_scores = loaded.decision_function(X[8:])
        self.assertEqual(loaded.features.hydra_device_, "cpu")
        self.assertEqual(cpu_scores.shape, (4,))
        self.assertTrue(np.isfinite(cpu_scores).all())
        self.assertNotIn("_hydra_device_transform_", loaded.features.__dict__)
        # The same fitted scaler and classifier serve CPU inference; CUDA's
        # convolution and scatter order need not reproduce bit-identical scores.


if __name__ == "__main__":
    unittest.main()
'''
destination = WORK / 'sota_pd_et/tests/test_gpu.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(embedded_source, encoding='utf-8')
print('Prepared:', destination.relative_to(WORK))


## 4. Verify the pipeline on small synthetic data

These tests check diagnosis selection, checksum corruption, finite
values, train/test separation, fold-local feature fitting, metrics,
model serialization, the complete nested-CV/prediction path, and the
selective downloader. Device selection, exact CPU adapter equivalence,
portable model saving, and GPU smoke checks are included; actual CUDA
checks run only when CUDA is available. Synthetic checks do not measure real-data
predictive performance. They are enabled by default.


In [ ]:
if RUN_SYNTHETIC_CHECKS:
    run_command([PYTHON, '-m', 'unittest', 'discover', '-s',
                 str(WORK / 'sota_pd_et' / 'tests'), '-v'],
                log_path=WORK / 'synthetic_checks.log')
else:
    print('Synthetic checks skipped by configuration.')


### Optional GPU timing check

If CUDA is available, the next cell compares the CPU and GPU HYDRA
transforms on eight synthetic inputs with the PADS dimensions. It
reports warmed-up transform times **including data transfers**, and the
maximum CPU/GPU feature difference. It does not fit or benchmark the
complete classifier. A ratio above one means the GPU was faster for
this small check; it does not predict the full experiment's speedup.
`RUN_GPU_BENCHMARK = False` skips it. CPU runtimes skip it automatically.


In [ ]:
if RUN_GPU_BENCHMARK and HYDRA_DEVICE != 'cpu':
    benchmark_code = r'''
import json, sys, time
from pathlib import Path
import numpy as np
import torch
from aeon.transformations.collection.convolution_based import HydraTransformer
from sota_pd_et.hydra_gpu import HydraDeviceTransform

destination = Path(sys.argv[1])
if not torch.cuda.is_available():
    result = {'status': 'skipped', 'reason': 'CUDA is unavailable.'}
else:
    samples = np.random.default_rng(17).normal(size=(8, 132, 976)).astype(np.float32)
    reference = HydraTransformer(n_kernels=8, n_groups=64, n_jobs=int(sys.argv[2]),
                                 random_state=17, output_type='tensor').fit(samples)
    accelerated = HydraDeviceTransform(reference, 'cuda')
    # Warm both paths before measuring; weights are fitted only once.
    with torch.inference_mode():
        expected = reference.transform(samples)
        observed = accelerated.transform(samples)
        timings = {}
        for name, operation in [('cpu', reference.transform), ('cuda', accelerated.transform)]:
            times = []
            for _ in range(3):
                torch.cuda.synchronize()
                start = time.perf_counter()
                output = operation(samples)
                torch.cuda.synchronize()
                times.append(time.perf_counter() - start)
            timings[name] = float(np.median(times))
    result = {'status': 'complete', 'gpu': torch.cuda.get_device_name(0),
              'shape': list(samples.shape), 'features_per_participant': int(expected.shape[1]),
              'cpu_seconds_median': timings['cpu'], 'cuda_seconds_median': timings['cuda'],
              'cpu_time_divided_by_cuda_time': timings['cpu'] / timings['cuda'],
              'max_absolute_feature_difference': float((expected - observed).abs().max()),
              'scope': 'Synthetic HYDRA transform only, including transfers; not whole training.'}
destination.write_text(json.dumps(result, indent=2) + '\n')
print(json.dumps(result, indent=2))
'''
    # Remove notebook indentation from the embedded benchmark program.
    import textwrap
    run_command([PYTHON, '-c', textwrap.dedent(benchmark_code),
                 WORK / 'gpu_benchmark.json', str(N_JOBS)])
else:
    print('GPU timing check skipped by configuration.')


## 5. Download and verify the official classification inputs

Download the versioned official checksum list, participant table, two
upstream preprocessing scripts, and each selected participant's JSON
and movement binary. The checksum list itself is pinned by its known
SHA-256; every selected file must match its official entry. A rerun
reuses only files whose hashes still match. Interrupted partial files
are downloaded again. Nothing extracts or downloads the full archive.

The binary input is the authors' supplied movement representation:
published task selection, gravity correction, and cropping have already
been applied. This workflow does not resample raw timestamps or correct
raw recording gaps. Questionnaire, demographic, ID, and disease-comment
fields are not model features. Published diagnosis assignments are
retained, including PD-assigned participants whose comments mention ET.


In [ ]:
run_command([PYTHON, '-m', 'sota_pd_et.colab_download',
             DATA_ROOT, '--workers', str(DOWNLOAD_WORKERS)],
            log_path=WORK / 'data_download.log')
run_command([PYTHON, '-c',
             'import json, sys; from sota_pd_et.data import load_cohort; '
             'c = load_cohort(sys.argv[1]); '
             'print(json.dumps({"shape": list(c.X.shape), '
             '"counts": c.provenance["condition_counts"], '
             '"verified_sources": len(c.provenance["verified_source_sha256"])}, indent=2)); '
             'assert c.X.shape == (304, 132, 976); '
             'assert (int((c.y == 0).sum()), int((c.y == 1).sum())) == (276, 28)',
             DATA_ROOT])


## 6. Run participant-level nested cross-validation and fit the final model

For outer fold $j$, the training participants alone enter an inner
stratified cross-validation. In each inner fold, fit **fresh HYDRA and
MultiROCKET transforms and their scalers** using only that inner training
set. Fit a class-balanced ridge classifier for each candidate $\alpha$.
Choose the largest $\alpha$ among mean inner balanced accuracies tied
within $10^{-12}$. Refit the full feature map and classifier on the outer
training participants, then evaluate that outer test fold once.

The positive class is **ET**, and the fixed rule is **ET if the signed
ridge margin is strictly greater than zero; PD otherwise**. Class weights
in each ridge fit are $w_c=n_{\mathrm{train}}/(2n_c)$, where $n_c$ is the
training count of class $c\in\{0,1\}$. Balanced accuracy is
$(\mathrm{ET\ recall}+\mathrm{PD\ recall})/2$. AUROC and average precision
use continuous margins, which are **not probabilities**.

Each participant has exactly one outer test prediction per repeat.
Repeats reuse the same people. The optional final fit retunes alpha on
the full cohort and fits all participants; it has no independent test
set. Its training predictions are not reported as evaluation results.

The default configuration fits **24 feature maps** including the final
model. Progress is streamed below and saved to a log. Each execution
creates a new result directory. Interrupted runs retain completed fold
files, but **automatic training resumption is not implemented**; rerun
this cell to start a fresh experiment.


In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = WORK / 'results' / ('run_' + stamp)
train_command = [
    PYTHON, '-m', 'sota_pd_et.evaluate',
    '--data-root', DATA_ROOT, '--output', RUN_DIR,
    '--outer-folds', str(OUTER_FOLDS), '--inner-folds', str(INNER_FOLDS),
    '--repeats', str(REPEATS), '--seed', str(SEED),
    '--n-jobs', str(N_JOBS), '--batch-size', str(BATCH_SIZE),
    '--mr-kernels', str(MR_KERNELS), '--hydra-groups', str(HYDRA_GROUPS),
    '--hydra-kernels', str(HYDRA_KERNELS), '--hydra-device', HYDRA_DEVICE,
    '--alphas', *[str(alpha) for alpha in ALPHAS],
]
if FIT_FINAL_MODEL:
    train_command.append('--fit-final')
print('Starting experiment:', RUN_DIR.name, flush=True)
run_command(train_command, log_path=WORK / (RUN_DIR.name + '_training.log'))
assert json.loads((RUN_DIR / 'manifest.json').read_text())['status'] == 'complete'
print('Completed:', RUN_DIR)


## 7. Inspect held-out results

The report verifies participant coverage and recalculates the metrics
from saved outer predictions before plotting. Confusion-matrix rows are
true diagnoses and columns are predictions, both ordered **[PD, ET]**.
ROC and PR panels include fold curves and pooled curves. Pooled ranking
metrics compare margins from separately fitted models, whose scales can
differ; also inspect the per-fold AUROC and AP. Fold and repeat standard
deviations are descriptive, not confidence intervals.

An always-PD classifier has balanced accuracy 0.5 and ET recall zero,
despite ordinary accuracy approximately 0.908. The ET average-precision
no-skill reference is the ET prevalence, approximately 0.0921. With only
28 ET participants, inspect class counts and errors alongside metrics.


In [ ]:
REPORT_DIR = RUN_DIR / 'report'
run_command([PYTHON, '-m', 'sota_pd_et.colab_report',
             '--run', RUN_DIR, '--output', REPORT_DIR])
summary = json.loads((RUN_DIR / 'summary.json').read_text())
keys = ['balanced_accuracy', 'accuracy', 'roc_auc', 'average_precision_et',
        'et_recall', 'pd_recall', 'et_precision', 'f1_et',
        'macro_f1', 'matthews_correlation']
heading = '<tr><th>Metric</th>' + ''.join(
    '<th>Repeat ' + str(row['repeat']) + '</th>'
    for row in summary['repeat_metrics']) + '<th>Always PD</th></tr>'
table_rows = ''.join(
    '<tr><td>' + html.escape(key) + '</td>' + ''.join(
        '<td>' + format(row[key], '.4f') + '</td>'
        for row in summary['repeat_metrics']) + '<td>' +
    format(summary['always_predict_pd_baseline'][key], '.4f') + '</td></tr>'
    for key in keys)
display(HTML('<table>' + heading + table_rows + '</table>'))
for figure in sorted(REPORT_DIR.glob('*.png')):
    display(Markdown('**' + figure.stem.replace('_', ' ') + '**'))
    display(Image(filename=str(figure)))
print('Fold and repeat metrics:', RUN_DIR / 'summary.json')
for key in ('roc_auc', 'average_precision_et'):
    result = summary['across_outer_folds_descriptive'][key]
    print('Outer-fold', key, 'descriptive mean:', result['mean'],
          '| descriptive SD:', result['std'])


## 8. Optional inference on new, already preprocessed inputs

This cell is skipped unless `NEW_INPUTS_NPY` is set in the configuration.
Input must be a numeric `.npy` array of shape **(N, 132, 976)** with the
**same upstream preprocessing and exact channel order** as PADS. An
arbitrary raw recording cannot be passed directly. Copy/upload your
file into the runtime and set its absolute path before running this
cell. The saved model records the channel names and training IDs.

Use only a trusted model file that you produced yourself. The prediction
entry point checks the installed model package versions against the
training bundle. Inference for a training participant is not a held-out
evaluation. Positive scores favor ET but are not calibrated probabilities.


In [ ]:
if NEW_INPUTS_NPY is None:
    print('New-input inference skipped; NEW_INPUTS_NPY is None.')
else:
    model_path = RUN_DIR / 'final_model.joblib'
    if not model_path.is_file():
        raise FileNotFoundError('Fit the final model before running new-input inference.')
    inputs_path = Path(NEW_INPUTS_NPY).expanduser().resolve()
    if not inputs_path.is_file():
        raise FileNotFoundError(inputs_path)
    prediction_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    prediction_path = RUN_DIR / ('new_predictions_' + prediction_stamp + '.csv')
    run_command([PYTHON, '-m', 'sota_pd_et.predict', '--model', model_path,
                 '--inputs', inputs_path, '--output', prediction_path,
                 '--hydra-device', HYDRA_DEVICE])
    with prediction_path.open(newline='') as stream:
        for row in csv.DictReader(stream):
            print(row)


## 9. Download the results and the reusable code

The ZIP contains this run's predictions, exact splits, configuration and
source/data hashes, metrics, figures, report, actual package lock,
embedded package, tests, logs, and optional final model. It excludes the
downloaded dataset and the installed environment. The source code is
necessary to reload the saved model in another session. Recreate the
recorded environment before reusing a model. `torch-install.json`
records the selected CPU or CUDA wheel index. GPU execution caches
are excluded from saved models, allowing CPU inference with the same
package versions when a GPU is unavailable.

The browser download starts automatically when `AUTO_DOWNLOAD = True`.
The Colab Files sidebar also exposes the ZIP under
`/content/pd_et_colab/`. If you deliberately stop training, run
this export cell manually to recover its partial results; these must
not be reported as a complete cross-validation experiment.


In [ ]:
if 'RUN_DIR' not in globals() or not RUN_DIR.is_dir():
    raise RuntimeError('No experiment directory exists to export yet.')
run_manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
note = '''PADS PD-versus-ET standalone notebook results

Data source: https://physionet.org/content/parkinsons-disease-smartwatch/1.0.0/
Data DOI: https://doi.org/10.13026/m0w9-zx22
Data and data derivatives: CC BY-NC-SA 4.0; follow official attribution.

results/ contains the experiment outputs. Check manifest.json status
before using metrics. Inner tuning outputs are not outer evaluation.
final_model.joblib, when present, was trained on all cohort participants.
Ridge scores favor ET when strictly positive and are not probabilities.

Reuse: keep the sota_pd_et/ directory beside your inference code, restore
environment-lock.txt in a compatible Python environment, and inspect the
Python/platform values in results/manifest.json. On Linux, install the
recorded torch version using the index saved in torch-install.json:
  python -m pip install torch==VERSION --index-url INDEX_URL
  python -m pip install -r environment-lock.txt
  python -m sota_pd_et.predict --model results/final_model.joblib --inputs new.npy --output predictions.csv
Replace VERSION with the torch version recorded in environment-lock.txt.
Replace INDEX_URL with index_url from torch-install.json. CUDA builds
also allow CPU inference; use --hydra-device cpu to force it. No CUDA
execution cache is serialized. CPU/CUDA rounding may differ near ties.
New inputs must use the identical PADS preprocessing/channel order.
Load only model files you trust. No source dataset is included here.
'''
note = '\n'.join(line[8:] if line.startswith('        ') else line
                 for line in note.splitlines()) + '\n'
archive = WORK / (RUN_DIR.name + '_bundle.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.writestr('README.txt', note)
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file():
            bundle.write(path, str(Path('results') / path.relative_to(RUN_DIR)))
    for path in sorted((WORK / 'sota_pd_et').rglob('*.py')):
        bundle.write(path, str(path.relative_to(WORK)))
    for pattern in ('environment-lock.txt', 'requirements-colab.txt', 'torch-install.json', 'gpu_benchmark.json',
                    'synthetic_checks.log', 'data_download.log',
                    RUN_DIR.name + '_training.log'):
        path = WORK / pattern
        if path.is_file():
            bundle.write(path, path.name)
print('Experiment status:', run_manifest['status'])
print('ZIP:', archive, '| bytes:', archive.stat().st_size)
print('ZIP SHA-256:', hashlib.sha256(archive.read_bytes()).hexdigest())
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(archive)))
else:
    if AUTO_DOWNLOAD:
        files.download(str(archive))
    else:
        print('Download the ZIP from the Colab Files sidebar:', archive)


## References and interpretation

- [Official PADS v1.0.0 dataset and citation instructions](https://physionet.org/content/parkinsons-disease-smartwatch/1.0.0/).
- [HYDRA](https://doi.org/10.1007/s10618-023-00939-3) and
  [MultiROCKET](https://doi.org/10.1007/s10618-022-00844-1).
- [aeon 1.6.0 MultiROCKET–HYDRA implementation](https://github.com/aeon-toolkit/aeon/blob/v1.6.0/aeon/classification/convolution_based/_mr_hydra.py)
  and [package compatibility declaration](https://github.com/aeon-toolkit/aeon/blob/v1.6.0/pyproject.toml).
- [Official PyTorch installation](https://pytorch.org/get-started/locally/)
  and [CUDA numerical behavior](https://docs.pytorch.org/docs/stable/notes/cuda.html).
- [Google Colab resource and runtime FAQ](https://research.google.com/colaboratory/faq.html).

The transforms come from aeon. Alpha selection is explicitly nested
around the entire fitted feature map. This estimates the specified
procedure on the published PADS representation. It does not establish
external generalization or a clinical decision rule. Changing a
threshold after seeing outer test labels and then reporting performance
on those same labels would no longer be an independent evaluation.
